# Franklin Pool: tip-split payout math

Split a coffee shop's tip pool (20% of online-order revenue) across staff by minutes worked, in whole cents, with the largest-remainder method. There is exactly one right answer per case, and a case passes only if every worker's cents match.

The model answers directly, with no tools.

220 cases in 11 categories. 40 of them turn on an email tie-break that locale-aware sort, natural sort and table order all get wrong. Source and answer key: franklin-pool-bench (`generate.mjs`, seed 20260923).

In [ ]:
import json
import os
import time

import pandas as pd
import kaggle_benchmarks as kbench

In [ ]:
# Answer key, prompt builder and grader (franklin_core.py).
import json
import subprocess
import sys

FUNDING_SOURCES = {"online", "agent_api"}


def minutes_of(w):
    return max(0, w["regular"] + w["overtime"] + w["sunday_regular"] + w["sunday_overtime"])


def allocate(workers, orders):
    """Every worker's cents (0 if excluded). Python ints never lose precision."""
    subtotal = 0
    for o in orders:
        if str(o["source"] if o["source"] is not None else "").strip().lower() not in FUNDING_SOURCES:
            continue
        subtotal += max(0, round(o["subtotal_cents"]))
    pool = subtotal * 2000 // 10000

    out = {w["email"]: 0 for w in workers}
    active = [(w["email"], minutes_of(w)) for w in workers if minutes_of(w) > 0]
    total = sum(m for _, m in active)
    if pool <= 0 or total <= 0:
        return out
    shares = [[e, m * pool // total, m * pool % total] for e, m in active]
    leftover = pool - sum(s[1] for s in shares)
    shares.sort(key=lambda s: (-s[2], s[0]))  # str order is plain code-point order
    for s in shares[:leftover]:
        s[1] += 1
    for e, cents, _ in shares:
        out[e] = cents
    return out


def build_prompt(rules, workers, orders):
    wl = "\n".join(
        f"| {w['email']} | {w['regular']} | {w['overtime']} | {w['sunday_regular']} | {w['sunday_overtime']} |"
        for w in workers
    )
    if orders:
        ol = "\n".join(
            f"| {'(null)' if o['source'] is None else json.dumps(o['source'], ensure_ascii=False)} | {o['subtotal_cents']} |"
            for o in orders
        )
    else:
        ol = "| (no orders) | |"
    return f"""{rules}

## Workers (minutes)
| email | regular | overtime | sunday_regular | sunday_overtime |
|---|---|---|---|---|
{wl}

## Orders
| source | subtotal_cents |
|---|---|
{ol}

Reply with only a JSON object mapping every worker email above to their integer cents, e.g. {{"a@x.com": 120, "b@x.com": 0}}."""


def _reject_constant(name):
    raise ValueError(name)


def _is_answer(o):
    return (
        isinstance(o, dict) and len(o) > 0
        and all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in o.values())
    )


def _matching_brace(text, start):
    depth, in_string, i = 0, False, start
    while i < len(text):
        ch = text[i]
        if in_string:
            if ch == "\\":
                i += 1
            elif ch == '"':
                in_string = False
        elif ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return i
        i += 1
    return -1


def parse_output(text):
    """The last balanced {...} block that parses as a JSON object of numbers, or None."""
    found, found_end = None, -1
    start = text.find("{")
    while start >= 0:
        end = _matching_brace(text, start)
        if end >= 0:
            try:
                obj = json.loads(text[start : end + 1], parse_constant=_reject_constant)
            except ValueError:
                obj = None
            if _is_answer(obj) and end > found_end:
                found, found_end = obj, end
        start = text.find("{", start + 1)
    return found


def grade(answer, got):
    """(passed, sums_to_pool) for a parsed output against the answer key."""
    if got is None:
        return False, False
    passed = all(got.get(e) == v and not isinstance(got.get(e), bool) for e, v in answer.items())
    got_sum = sum(got[e] for e in answer if isinstance(got.get(e), (int, float)) and not isinstance(got.get(e), bool))
    return passed, got_sum == sum(answer.values())


def run_python_script(code: str) -> str:
    """Run a Python 3 script and return what it prints (stdout, then stderr). Use print() to see values. 30 second limit."""
    try:
        p = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=30)
    except subprocess.TimeoutExpired:
        return "error: timed out after 30 seconds"
    return (p.stdout + p.stderr)[-8000:] or "(no output)"

In [ ]:
RULES = "# Task: split a tip pool in integer cents\n\nA coffee shop shares 20% of its online-order revenue with staff in proportion to minutes worked. Compute each worker's payout in whole cents using these rules exactly.\n\n1. **Funding.** An order counts only if its `source`, after trimming whitespace and lowercasing, is exactly `online` or `agent_api`. Any other source, including a missing (null) one, does not count. Each counted order contributes `max(0, subtotal_cents)`.\n2. **Pool.** `pool = floor(total_counted_subtotal × 2000 / 10000)` cents (20%, rounded down).\n3. **Minutes.** A worker's minutes = regular + overtime + sunday_regular + sunday_overtime. Workers with 0 minutes get 0 cents and are left out of every step below. `team_minutes` = the sum over the remaining workers.\n4. **If the pool is 0 or no one has minutes,** everyone gets 0.\n5. **Base share.** For each worker, `base = floor(minutes × pool / team_minutes)` and `remainder = (minutes × pool) mod team_minutes` (an integer).\n6. **Leftover.** `leftover = pool − sum of base shares`. Sort workers by `remainder` descending; break exact ties by email ascending in plain character order — compare character codes left to right (ASCII: `+` < `-` < `.` < digits < `@` < `_` < lowercase letters), a shorter string sorts first if it is a prefix of the longer one. No locale, natural-number or case-insensitive ordering. Give 1 extra cent to each of the first `leftover` workers in that order.\n7. The payouts must add up to exactly `pool`."

CASES = json.loads("[{\"id\":\"even_split-000\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"lu@brewhubphl.com\":2796,\"oz@brewhubphl.com\":2796,\"nia42@brewhubphl.com\":2796},\"workers\":[{\"email\":\"lu@brewhubphl.com\",\"regular\":165,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":165,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia42@brewhubphl.com\",\"regular\":165,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":41943}]},{\"id\":\"even_split-001\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"eli@brewhubphl.com\":3661,\"nia@brewhubphl.com\":3661,\"oz@brewhubphl.com\":3661,\"lu@brewhubphl.com\":3661,\"mo45@brewhubphl.com\":3661,\"gus@brewhubphl.com\":3661},\"workers\":[{\"email\":\"eli@brewhubphl.com\",\"regular\":1140,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia@brewhubphl.com\",\"regular\":1140,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":1140,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":1140,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo45@brewhubphl.com\",\"regular\":1140,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":1140,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":109833}]},{\"id\":\"even_split-002\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"fay@brewhubphl.com\":1504,\"ben@brewhubphl.com\":1504},\"workers\":[{\"email\":\"fay@brewhubphl.com\",\"regular\":1509,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":1509,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":15040}]},{\"id\":\"even_split-003\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"jo@brewhubphl.com\":3879,\"mo@brewhubphl.com\":3879,\"dee@brewhubphl.com\":3879,\"cy@brewhubphl.com\":3879},\"workers\":[{\"email\":\"jo@brewhubphl.com\",\"regular\":1877,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":1877,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":1877,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy@brewhubphl.com\",\"regular\":1877,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":77582}]},{\"id\":\"even_split-004\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"lu@brewhubphl.com\":3107,\"mo@brewhubphl.com\":3107,\"ben@brewhubphl.com\":3107},\"workers\":[{\"email\":\"lu@brewhubphl.com\",\"regular\":1659,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":1659,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":1659,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":46609}]},{\"id\":\"even_split-005\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"pia@brewhubphl.com\":2394,\"ivo@brewhubphl.com\":2394,\"oz@brewhubphl.com\":2394,\"cy9@brewhubphl.com\":2394,\"eli74@brewhubphl.com\":2394,\"dee80@brewhubphl.com\":2394},\"workers\":[{\"email\":\"pia@brewhubphl.com\",\"regular\":448,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo@brewhubphl.com\",\"regular\":448,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":448,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy9@brewhubphl.com\",\"regular\":448,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli74@brewhubphl.com\",\"regular\":448,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee80@brewhubphl.com\",\"regular\":448,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":71822}]},{\"id\":\"even_split-006\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"hana@brewhubphl.com\":129,\"kai66@brewhubphl.com\":129},\"workers\":[{\"email\":\"hana@brewhubphl.com\",\"regular\":281,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai66@brewhubphl.com\",\"regular\":281,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1293}]},{\"id\":\"even_split-007\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"pia@brewhubphl.com\":2485,\"jo18@brewhubphl.com\":2485,\"nia@brewhubphl.com\":2485,\"gus@brewhubphl.com\":2485},\"workers\":[{\"email\":\"pia@brewhubphl.com\",\"regular\":265,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo18@brewhubphl.com\",\"regular\":265,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia@brewhubphl.com\",\"regular\":265,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":265,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":49704}]},{\"id\":\"even_split-008\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"jo@brewhubphl.com\":3592,\"gus@brewhubphl.com\":3592},\"workers\":[{\"email\":\"jo@brewhubphl.com\",\"regular\":1251,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":1251,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":35920}]},{\"id\":\"even_split-009\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"lu@brewhubphl.com\":1688,\"hana@brewhubphl.com\":1688},\"workers\":[{\"email\":\"lu@brewhubphl.com\",\"regular\":565,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":565,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":16881}]},{\"id\":\"even_split-010\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"kai@brewhubphl.com\":640,\"nia@brewhubphl.com\":640,\"ivo@brewhubphl.com\":640},\"workers\":[{\"email\":\"kai@brewhubphl.com\",\"regular\":2068,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia@brewhubphl.com\",\"regular\":2068,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo@brewhubphl.com\",\"regular\":2068,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":9603}]},{\"id\":\"even_split-011\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"eli@brewhubphl.com\":4736,\"jo31@brewhubphl.com\":4736,\"ben32@brewhubphl.com\":4736,\"ivo@brewhubphl.com\":4736},\"workers\":[{\"email\":\"eli@brewhubphl.com\",\"regular\":981,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo31@brewhubphl.com\",\"regular\":981,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben32@brewhubphl.com\",\"regular\":981,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo@brewhubphl.com\",\"regular\":981,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":94720}]},{\"id\":\"even_split-012\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"cy@brewhubphl.com\":4877,\"hana@brewhubphl.com\":4877,\"ivo@brewhubphl.com\":4877,\"fay@brewhubphl.com\":4877,\"jo@brewhubphl.com\":4877,\"nia@brewhubphl.com\":4877},\"workers\":[{\"email\":\"cy@brewhubphl.com\",\"regular\":2160,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":2160,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo@brewhubphl.com\",\"regular\":2160,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":2160,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":2160,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia@brewhubphl.com\",\"regular\":2160,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":146310}]},{\"id\":\"even_split-013\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"nia6@brewhubphl.com\":1596,\"fay26@brewhubphl.com\":1596},\"workers\":[{\"email\":\"nia6@brewhubphl.com\",\"regular\":2338,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay26@brewhubphl.com\",\"regular\":2338,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":15960}]},{\"id\":\"even_split-014\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"jo67@brewhubphl.com\":2704,\"kai21@brewhubphl.com\":2704,\"dee@brewhubphl.com\":2704,\"oz@brewhubphl.com\":2704},\"workers\":[{\"email\":\"jo67@brewhubphl.com\",\"regular\":419,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai21@brewhubphl.com\",\"regular\":419,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":419,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":419,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":54080}]},{\"id\":\"even_split-015\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"fay91@brewhubphl.com\":3905,\"dee88@brewhubphl.com\":3905},\"workers\":[{\"email\":\"fay91@brewhubphl.com\",\"regular\":1877,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee88@brewhubphl.com\",\"regular\":1877,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":39050}]},{\"id\":\"even_split-016\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"lu20@brewhubphl.com\":2674,\"nia45@brewhubphl.com\":2674,\"gus@brewhubphl.com\":2674,\"dee@brewhubphl.com\":2674,\"pia@brewhubphl.com\":2674},\"workers\":[{\"email\":\"lu20@brewhubphl.com\",\"regular\":1356,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia45@brewhubphl.com\",\"regular\":1356,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":1356,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":1356,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":1356,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":66853}]},{\"id\":\"even_split-017\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"fay@brewhubphl.com\":2835,\"jo@brewhubphl.com\":2835,\"ben19@brewhubphl.com\":2835},\"workers\":[{\"email\":\"fay@brewhubphl.com\",\"regular\":399,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":399,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben19@brewhubphl.com\",\"regular\":399,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":42527}]},{\"id\":\"even_split-018\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"ivo@brewhubphl.com\":2104,\"ana43@brewhubphl.com\":2104,\"cy@brewhubphl.com\":2104,\"ana57@brewhubphl.com\":2104},\"workers\":[{\"email\":\"ivo@brewhubphl.com\",\"regular\":1795,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana43@brewhubphl.com\",\"regular\":1795,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy@brewhubphl.com\",\"regular\":1795,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana57@brewhubphl.com\",\"regular\":1795,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":42083}]},{\"id\":\"even_split-019\",\"category\":\"even_split\",\"tie_decides\":false,\"answer\":{\"gus41@brewhubphl.com\":4809,\"lu@brewhubphl.com\":4809,\"dee@brewhubphl.com\":4809,\"dee42@brewhubphl.com\":4809,\"hana15@brewhubphl.com\":4809,\"oz31@brewhubphl.com\":4809},\"workers\":[{\"email\":\"gus41@brewhubphl.com\",\"regular\":1561,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":1561,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":1561,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee42@brewhubphl.com\",\"regular\":1561,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana15@brewhubphl.com\",\"regular\":1561,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz31@brewhubphl.com\",\"regular\":1561,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":144274}]},{\"id\":\"simple_remainder-000\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"ana@brewhubphl.com\":1248,\"cy75@brewhubphl.com\":16311},\"workers\":[{\"email\":\"ana@brewhubphl.com\",\"regular\":210,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy75@brewhubphl.com\",\"regular\":2744,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":87796}]},{\"id\":\"simple_remainder-001\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"oz33@brewhubphl.com\":21824,\"kai@brewhubphl.com\":21650,\"eli82@brewhubphl.com\":2965},\"workers\":[{\"email\":\"oz33@brewhubphl.com\",\"regular\":1759,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":1745,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli82@brewhubphl.com\",\"regular\":239,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":232197}]},{\"id\":\"simple_remainder-002\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"fay@brewhubphl.com\":8420,\"kai@brewhubphl.com\":5107,\"ana85@brewhubphl.com\":2066,\"hana89@brewhubphl.com\":2739},\"workers\":[{\"email\":\"fay@brewhubphl.com\",\"regular\":1365,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":828,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana85@brewhubphl.com\",\"regular\":335,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana89@brewhubphl.com\",\"regular\":444,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":91662}]},{\"id\":\"simple_remainder-003\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"gus65@brewhubphl.com\":8164,\"ivo@brewhubphl.com\":23222,\"cy@brewhubphl.com\":18729},\"workers\":[{\"email\":\"gus65@brewhubphl.com\",\"regular\":656,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo@brewhubphl.com\",\"regular\":1866,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy@brewhubphl.com\",\"regular\":1505,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":250575}]},{\"id\":\"simple_remainder-004\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"eli@brewhubphl.com\":17259,\"gus40@brewhubphl.com\":5467,\"cy@brewhubphl.com\":18547,\"jo@brewhubphl.com\":12536},\"workers\":[{\"email\":\"eli@brewhubphl.com\",\"regular\":2090,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus40@brewhubphl.com\",\"regular\":662,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy@brewhubphl.com\",\"regular\":2246,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":1518,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":269049}]},{\"id\":\"simple_remainder-005\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"pia49@brewhubphl.com\":10883,\"kai@brewhubphl.com\":6993,\"pia83@brewhubphl.com\":614},\"workers\":[{\"email\":\"pia49@brewhubphl.com\",\"regular\":1614,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":1037,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia83@brewhubphl.com\",\"regular\":91,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":92451}]},{\"id\":\"simple_remainder-006\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"ana@brewhubphl.com\":19955,\"kai@brewhubphl.com\":31348,\"pia@brewhubphl.com\":14496,\"nia77@brewhubphl.com\":888,\"lu@brewhubphl.com\":16002},\"workers\":[{\"email\":\"ana@brewhubphl.com\",\"regular\":1550,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":2435,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":1126,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia77@brewhubphl.com\",\"regular\":69,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":1243,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":413446}]},{\"id\":\"simple_remainder-007\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"mo@brewhubphl.com\":7693,\"gus@brewhubphl.com\":20913,\"jo15@brewhubphl.com\":22357},\"workers\":[{\"email\":\"mo@brewhubphl.com\",\"regular\":522,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":1419,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo15@brewhubphl.com\",\"regular\":1517,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":254817}]},{\"id\":\"simple_remainder-008\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"lu57@brewhubphl.com\":78485,\"ana66@brewhubphl.com\":10102},\"workers\":[{\"email\":\"lu57@brewhubphl.com\",\"regular\":2525,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana66@brewhubphl.com\",\"regular\":325,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":442937}]},{\"id\":\"simple_remainder-009\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"mo@brewhubphl.com\":23173,\"fay18@brewhubphl.com\":14095,\"kai@brewhubphl.com\":17861,\"lu@brewhubphl.com\":13928},\"workers\":[{\"email\":\"mo@brewhubphl.com\",\"regular\":2504,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay18@brewhubphl.com\",\"regular\":1523,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":1930,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":1505,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":345288}]},{\"id\":\"simple_remainder-010\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"eli@brewhubphl.com\":6074,\"mo31@brewhubphl.com\":3920,\"mo@brewhubphl.com\":5780,\"lu62@brewhubphl.com\":2537},\"workers\":[{\"email\":\"eli@brewhubphl.com\",\"regular\":2997,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo31@brewhubphl.com\",\"regular\":1934,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":2852,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu62@brewhubphl.com\",\"regular\":1252,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":91559}]},{\"id\":\"simple_remainder-011\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"pia5@brewhubphl.com\":2733,\"eli@brewhubphl.com\":5610,\"hana@brewhubphl.com\":840,\"gus@brewhubphl.com\":6369,\"pia@brewhubphl.com\":2219},\"workers\":[{\"email\":\"pia5@brewhubphl.com\",\"regular\":1067,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli@brewhubphl.com\",\"regular\":2190,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":328,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":2486,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":866,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":88857}]},{\"id\":\"simple_remainder-012\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"mo@brewhubphl.com\":8121,\"ana72@brewhubphl.com\":5366,\"cy@brewhubphl.com\":11101,\"fay21@brewhubphl.com\":6450},\"workers\":[{\"email\":\"mo@brewhubphl.com\",\"regular\":1147,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana72@brewhubphl.com\",\"regular\":758,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy@brewhubphl.com\",\"regular\":1568,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay21@brewhubphl.com\",\"regular\":911,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":155194}]},{\"id\":\"simple_remainder-013\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"eli@brewhubphl.com\":3256,\"ana@brewhubphl.com\":1789,\"pia@brewhubphl.com\":3979,\"kai@brewhubphl.com\":1377,\"jo@brewhubphl.com\":8175},\"workers\":[{\"email\":\"eli@brewhubphl.com\",\"regular\":870,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":478,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":1063,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":368,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":2184,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":92883}]},{\"id\":\"simple_remainder-014\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"pia92@brewhubphl.com\":4756,\"ivo@brewhubphl.com\":1241,\"mo81@brewhubphl.com\":6093},\"workers\":[{\"email\":\"pia92@brewhubphl.com\",\"regular\":1894,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo@brewhubphl.com\",\"regular\":494,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo81@brewhubphl.com\",\"regular\":2426,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":60453}]},{\"id\":\"simple_remainder-015\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"kai@brewhubphl.com\":397,\"eli@brewhubphl.com\":15769,\"jo24@brewhubphl.com\":19170},\"workers\":[{\"email\":\"kai@brewhubphl.com\",\"regular\":54,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli@brewhubphl.com\",\"regular\":2142,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo24@brewhubphl.com\",\"regular\":2604,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":176681}]},{\"id\":\"simple_remainder-016\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"lu@brewhubphl.com\":4415,\"cy@brewhubphl.com\":3332,\"ana79@brewhubphl.com\":230,\"fay@brewhubphl.com\":3717,\"kai@brewhubphl.com\":5054},\"workers\":[{\"email\":\"lu@brewhubphl.com\",\"regular\":1284,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy@brewhubphl.com\",\"regular\":969,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana79@brewhubphl.com\",\"regular\":67,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":1081,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":1470,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":83742}]},{\"id\":\"simple_remainder-017\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"jo38@brewhubphl.com\":35885,\"mo@brewhubphl.com\":35226},\"workers\":[{\"email\":\"jo38@brewhubphl.com\",\"regular\":2720,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":2670,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":355556}]},{\"id\":\"simple_remainder-018\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"cy27@brewhubphl.com\":89932,\"hana6@brewhubphl.com\":5683},\"workers\":[{\"email\":\"cy27@brewhubphl.com\",\"regular\":1725,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana6@brewhubphl.com\",\"regular\":109,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":478078}]},{\"id\":\"simple_remainder-019\",\"category\":\"simple_remainder\",\"tie_decides\":false,\"answer\":{\"dee56@brewhubphl.com\":37532,\"kai71@brewhubphl.com\":25021},\"workers\":[{\"email\":\"dee56@brewhubphl.com\",\"regular\":1038,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai71@brewhubphl.com\",\"regular\":692,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":312767}]},{\"id\":\"exact_ties-000\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"oz8@brewhubphl.com\":1713,\"oz.x@brewhubphl.com\":1713,\"lu@brewhubphl.com\":1713,\"lu+x@brewhubphl.com\":1713,\"oz_x@brewhubphl.com\":1712,\"oz-pos@brewhubphl.com\":1713,\"lu3@brewhubphl.com\":1713},\"workers\":[{\"email\":\"oz8@brewhubphl.com\",\"regular\":633,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz.x@brewhubphl.com\",\"regular\":633,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":633,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu+x@brewhubphl.com\",\"regular\":633,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz_x@brewhubphl.com\",\"regular\":633,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz-pos@brewhubphl.com\",\"regular\":633,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu3@brewhubphl.com\",\"regular\":633,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":59953}]},{\"id\":\"exact_ties-001\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"gus7@brewhubphl.com\":918,\"mo+k@brewhubphl.com\":918,\"mo@brewhubphl.com\":918,\"gus2@brewhubphl.com\":919,\"gus@brewhubphl.com\":918,\"mo+pos@brewhubphl.com\":918,\"mo+x@brewhubphl.com\":918},\"workers\":[{\"email\":\"gus7@brewhubphl.com\",\"regular\":337,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo+k@brewhubphl.com\",\"regular\":337,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":337,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus2@brewhubphl.com\",\"regular\":337,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":337,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo+pos@brewhubphl.com\",\"regular\":337,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo+x@brewhubphl.com\",\"regular\":337,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":32139}]},{\"id\":\"exact_ties-002\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"mo-pos@brewhubphl.com\":653,\"mo_x@brewhubphl.com\":652,\"hana+pos@brewhubphl.com\":653,\"hana+k@brewhubphl.com\":653,\"mo.x@brewhubphl.com\":653},\"workers\":[{\"email\":\"mo-pos@brewhubphl.com\",\"regular\":348,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo_x@brewhubphl.com\",\"regular\":348,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana+pos@brewhubphl.com\",\"regular\":348,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana+k@brewhubphl.com\",\"regular\":348,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo.x@brewhubphl.com\",\"regular\":348,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":16322}]},{\"id\":\"exact_ties-003\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"fay9@brewhubphl.com\":142,\"fay@brewhubphl.com\":142,\"gus_pos@brewhubphl.com\":142,\"fay_pos@brewhubphl.com\":142,\"fay+x@brewhubphl.com\":143,\"fay39@brewhubphl.com\":142,\"gus@brewhubphl.com\":142},\"workers\":[{\"email\":\"fay9@brewhubphl.com\",\"regular\":314,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":314,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus_pos@brewhubphl.com\",\"regular\":314,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay_pos@brewhubphl.com\",\"regular\":314,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay+x@brewhubphl.com\",\"regular\":314,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay39@brewhubphl.com\",\"regular\":314,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":314,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":4978}]},{\"id\":\"exact_ties-004\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"hana@brewhubphl.com\":134,\"oz56@brewhubphl.com\":134,\"oz.pos@brewhubphl.com\":134,\"hana+pos@brewhubphl.com\":135,\"hana+x@brewhubphl.com\":135,\"oz-pos@brewhubphl.com\":134},\"workers\":[{\"email\":\"hana@brewhubphl.com\",\"regular\":116,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz56@brewhubphl.com\",\"regular\":116,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz.pos@brewhubphl.com\",\"regular\":116,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana+pos@brewhubphl.com\",\"regular\":116,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana+x@brewhubphl.com\",\"regular\":116,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz-pos@brewhubphl.com\",\"regular\":116,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":4031}]},{\"id\":\"exact_ties-005\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"eli54@brewhubphl.com\":742,\"ivo59@brewhubphl.com\":741,\"ivo66@brewhubphl.com\":741,\"ivo-pos@brewhubphl.com\":741,\"ivo+pos@brewhubphl.com\":742,\"eli_k@brewhubphl.com\":742,\"eli.pos@brewhubphl.com\":742},\"workers\":[{\"email\":\"eli54@brewhubphl.com\",\"regular\":629,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo59@brewhubphl.com\",\"regular\":629,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo66@brewhubphl.com\",\"regular\":629,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo-pos@brewhubphl.com\",\"regular\":629,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo+pos@brewhubphl.com\",\"regular\":629,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli_k@brewhubphl.com\",\"regular\":629,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli.pos@brewhubphl.com\",\"regular\":629,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":25957}]},{\"id\":\"exact_ties-006\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"cy_k@brewhubphl.com\":839,\"cy85@brewhubphl.com\":840,\"cy-x@brewhubphl.com\":840},\"workers\":[{\"email\":\"cy_k@brewhubphl.com\",\"regular\":511,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy85@brewhubphl.com\",\"regular\":511,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy-x@brewhubphl.com\",\"regular\":511,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":12598}]},{\"id\":\"exact_ties-007\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"jo-x@brewhubphl.com\":173,\"mo-k@brewhubphl.com\":173,\"jo+k@brewhubphl.com\":174},\"workers\":[{\"email\":\"jo-x@brewhubphl.com\",\"regular\":530,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo-k@brewhubphl.com\",\"regular\":530,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo+k@brewhubphl.com\",\"regular\":530,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":2602}]},{\"id\":\"exact_ties-008\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"oz+pos@brewhubphl.com\":1038,\"oz20@brewhubphl.com\":1037,\"mo41@brewhubphl.com\":1038,\"oz@brewhubphl.com\":1037,\"oz-x@brewhubphl.com\":1037,\"mo@brewhubphl.com\":1038},\"workers\":[{\"email\":\"oz+pos@brewhubphl.com\",\"regular\":201,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz20@brewhubphl.com\",\"regular\":201,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo41@brewhubphl.com\",\"regular\":201,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":201,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz-x@brewhubphl.com\",\"regular\":201,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":201,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":31129}]},{\"id\":\"exact_ties-009\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"dee.x@brewhubphl.com\":1720,\"dee+x@brewhubphl.com\":1720,\"dee2@brewhubphl.com\":1720,\"ben-x@brewhubphl.com\":1721,\"ben.k@brewhubphl.com\":1721,\"ben+k@brewhubphl.com\":1721,\"ben_pos@brewhubphl.com\":1720},\"workers\":[{\"email\":\"dee.x@brewhubphl.com\",\"regular\":296,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee+x@brewhubphl.com\",\"regular\":296,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee2@brewhubphl.com\",\"regular\":296,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben-x@brewhubphl.com\",\"regular\":296,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben.k@brewhubphl.com\",\"regular\":296,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben+k@brewhubphl.com\",\"regular\":296,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben_pos@brewhubphl.com\",\"regular\":296,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":60218}]},{\"id\":\"exact_ties-010\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"kai_x@brewhubphl.com\":753,\"kai+pos@brewhubphl.com\":754,\"hana9@brewhubphl.com\":754,\"kai40@brewhubphl.com\":754},\"workers\":[{\"email\":\"kai_x@brewhubphl.com\",\"regular\":128,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai+pos@brewhubphl.com\",\"regular\":128,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana9@brewhubphl.com\",\"regular\":128,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai40@brewhubphl.com\",\"regular\":128,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":15076}]},{\"id\":\"exact_ties-011\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"ivo-pos@brewhubphl.com\":1970,\"cy.x@brewhubphl.com\":1970,\"ivo_x@brewhubphl.com\":1970,\"cy@brewhubphl.com\":1970,\"cy-x@brewhubphl.com\":1971,\"cy_x@brewhubphl.com\":1970,\"cy31@brewhubphl.com\":1970},\"workers\":[{\"email\":\"ivo-pos@brewhubphl.com\",\"regular\":248,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy.x@brewhubphl.com\",\"regular\":248,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo_x@brewhubphl.com\",\"regular\":248,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy@brewhubphl.com\",\"regular\":248,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy-x@brewhubphl.com\",\"regular\":248,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy_x@brewhubphl.com\",\"regular\":248,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy31@brewhubphl.com\",\"regular\":248,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":68958}]},{\"id\":\"exact_ties-012\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"kai4@brewhubphl.com\":988,\"kai@brewhubphl.com\":988,\"kai-pos@brewhubphl.com\":989,\"kai30@brewhubphl.com\":989},\"workers\":[{\"email\":\"kai4@brewhubphl.com\",\"regular\":61,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":61,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai-pos@brewhubphl.com\",\"regular\":61,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai30@brewhubphl.com\",\"regular\":61,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":19773}]},{\"id\":\"exact_ties-013\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"eli42@brewhubphl.com\":263,\"eli@brewhubphl.com\":263,\"eli_k@brewhubphl.com\":263,\"eli.x@brewhubphl.com\":264,\"eli+x@brewhubphl.com\":264},\"workers\":[{\"email\":\"eli42@brewhubphl.com\",\"regular\":449,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli@brewhubphl.com\",\"regular\":449,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli_k@brewhubphl.com\",\"regular\":449,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli.x@brewhubphl.com\",\"regular\":449,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli+x@brewhubphl.com\",\"regular\":449,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":6585}]},{\"id\":\"exact_ties-014\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"ben68@brewhubphl.com\":865,\"ben-x@brewhubphl.com\":866,\"ana_x@brewhubphl.com\":866,\"ben+pos@brewhubphl.com\":866,\"ben9@brewhubphl.com\":865,\"ana+k@brewhubphl.com\":866,\"ben@brewhubphl.com\":865},\"workers\":[{\"email\":\"ben68@brewhubphl.com\",\"regular\":659,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben-x@brewhubphl.com\",\"regular\":659,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana_x@brewhubphl.com\",\"regular\":659,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben+pos@brewhubphl.com\",\"regular\":659,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben9@brewhubphl.com\",\"regular\":659,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana+k@brewhubphl.com\",\"regular\":659,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":659,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":30296}]},{\"id\":\"exact_ties-015\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"lu@brewhubphl.com\":996,\"lu7@brewhubphl.com\":997,\"lu+x@brewhubphl.com\":997,\"lu31@brewhubphl.com\":997,\"lu99@brewhubphl.com\":997,\"lu+pos@brewhubphl.com\":997},\"workers\":[{\"email\":\"lu@brewhubphl.com\",\"regular\":586,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu7@brewhubphl.com\",\"regular\":586,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu+x@brewhubphl.com\",\"regular\":586,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu31@brewhubphl.com\",\"regular\":586,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu99@brewhubphl.com\",\"regular\":586,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu+pos@brewhubphl.com\",\"regular\":586,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":29905}]},{\"id\":\"exact_ties-016\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"nia-k@brewhubphl.com\":1124,\"cy_k@brewhubphl.com\":1125,\"nia+pos@brewhubphl.com\":1125},\"workers\":[{\"email\":\"nia-k@brewhubphl.com\",\"regular\":407,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy_k@brewhubphl.com\",\"regular\":407,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia+pos@brewhubphl.com\",\"regular\":407,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":16874}]},{\"id\":\"exact_ties-017\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"kai_k@brewhubphl.com\":1175,\"gus@brewhubphl.com\":1176,\"kai.x@brewhubphl.com\":1176,\"gus+k@brewhubphl.com\":1176},\"workers\":[{\"email\":\"kai_k@brewhubphl.com\",\"regular\":156,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":156,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai.x@brewhubphl.com\",\"regular\":156,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus+k@brewhubphl.com\",\"regular\":156,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":23516}]},{\"id\":\"exact_ties-018\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"pia_x@brewhubphl.com\":136,\"kai+pos@brewhubphl.com\":137,\"pia.x@brewhubphl.com\":137,\"pia98@brewhubphl.com\":137,\"kai4@brewhubphl.com\":137,\"kai_x@brewhubphl.com\":137,\"pia_pos@brewhubphl.com\":136},\"workers\":[{\"email\":\"pia_x@brewhubphl.com\",\"regular\":86,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai+pos@brewhubphl.com\",\"regular\":86,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia.x@brewhubphl.com\",\"regular\":86,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia98@brewhubphl.com\",\"regular\":86,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai4@brewhubphl.com\",\"regular\":86,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai_x@brewhubphl.com\",\"regular\":86,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia_pos@brewhubphl.com\",\"regular\":86,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":4788}]},{\"id\":\"exact_ties-019\",\"category\":\"exact_ties\",\"tie_decides\":true,\"answer\":{\"pia@brewhubphl.com\":1282,\"pia1@brewhubphl.com\":1282,\"pia12@brewhubphl.com\":1282,\"pia+x@brewhubphl.com\":1283,\"pia55@brewhubphl.com\":1282,\"pia.k@brewhubphl.com\":1282},\"workers\":[{\"email\":\"pia@brewhubphl.com\",\"regular\":273,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia1@brewhubphl.com\",\"regular\":273,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia12@brewhubphl.com\",\"regular\":273,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia+x@brewhubphl.com\",\"regular\":273,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia55@brewhubphl.com\",\"regular\":273,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia.k@brewhubphl.com\",\"regular\":273,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":38468}]},{\"id\":\"email_order_traps-000\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"mo1@brewhubphl.com\":111,\"jo@brewhubphl.com\":223,\"cy57@brewhubphl.com\":223,\"mo11@brewhubphl.com\":112},\"workers\":[{\"email\":\"mo1@brewhubphl.com\",\"regular\":779,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":1558,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy57@brewhubphl.com\",\"regular\":1558,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo11@brewhubphl.com\",\"regular\":779,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":3345}]},{\"id\":\"email_order_traps-001\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"a_b@brewhubphl.com\":898,\"a-b@brewhubphl.com\":899},\"workers\":[{\"email\":\"a_b@brewhubphl.com\",\"regular\":487,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"a-b@brewhubphl.com\",\"regular\":487,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":8986}]},{\"id\":\"email_order_traps-002\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"jo@brewhubphl.com\":103,\"jo+pos@brewhubphl.com\":104},\"workers\":[{\"email\":\"jo@brewhubphl.com\",\"regular\":529,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo+pos@brewhubphl.com\",\"regular\":529,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1038}]},{\"id\":\"email_order_traps-003\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"jo@brewhubphl.com\":579,\"jo+pos@brewhubphl.com\":580},\"workers\":[{\"email\":\"jo@brewhubphl.com\",\"regular\":230,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo+pos@brewhubphl.com\",\"regular\":230,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":5799}]},{\"id\":\"email_order_traps-004\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"mo1@brewhubphl.com\":977,\"mo11@brewhubphl.com\":978},\"workers\":[{\"email\":\"mo1@brewhubphl.com\",\"regular\":826,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo11@brewhubphl.com\",\"regular\":826,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":9775}]},{\"id\":\"email_order_traps-005\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"b1@brewhubphl.com\":292,\"ana@brewhubphl.com\":585,\"b10@brewhubphl.com\":293,\"ben@brewhubphl.com\":585},\"workers\":[{\"email\":\"b1@brewhubphl.com\",\"regular\":475,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":950,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"b10@brewhubphl.com\",\"regular\":475,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":950,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":8778}]},{\"id\":\"email_order_traps-006\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"a_b@brewhubphl.com\":51,\"a-b@brewhubphl.com\":52},\"workers\":[{\"email\":\"a_b@brewhubphl.com\",\"regular\":526,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"a-b@brewhubphl.com\",\"regular\":526,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":518}]},{\"id\":\"email_order_traps-007\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"mo95@brewhubphl.com\":455,\"kai@brewhubphl.com\":455,\"a_b@brewhubphl.com\":227,\"a-b@brewhubphl.com\":228},\"workers\":[{\"email\":\"mo95@brewhubphl.com\",\"regular\":1338,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":1338,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"a_b@brewhubphl.com\",\"regular\":669,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"a-b@brewhubphl.com\",\"regular\":669,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":6828}]},{\"id\":\"email_order_traps-008\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"ann@brewhubphl.com\":270,\"ann1@brewhubphl.com\":271},\"workers\":[{\"email\":\"ann@brewhubphl.com\",\"regular\":564,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ann1@brewhubphl.com\",\"regular\":564,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":2709}]},{\"id\":\"email_order_traps-009\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"mo1@brewhubphl.com\":653,\"mo11@brewhubphl.com\":654},\"workers\":[{\"email\":\"mo1@brewhubphl.com\",\"regular\":764,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo11@brewhubphl.com\",\"regular\":764,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":6539}]},{\"id\":\"email_order_traps-010\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"b1@brewhubphl.com\":887,\"b10@brewhubphl.com\":888},\"workers\":[{\"email\":\"b1@brewhubphl.com\",\"regular\":489,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"b10@brewhubphl.com\",\"regular\":489,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":8878}]},{\"id\":\"email_order_traps-011\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"b1@brewhubphl.com\":379,\"b10@brewhubphl.com\":380},\"workers\":[{\"email\":\"b1@brewhubphl.com\",\"regular\":232,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"b10@brewhubphl.com\",\"regular\":232,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":3797}]},{\"id\":\"email_order_traps-012\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"jo@brewhubphl.com\":183,\"jo+pos@brewhubphl.com\":184},\"workers\":[{\"email\":\"jo@brewhubphl.com\",\"regular\":760,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo+pos@brewhubphl.com\",\"regular\":760,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1837}]},{\"id\":\"email_order_traps-013\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"mo1@brewhubphl.com\":777,\"mo11@brewhubphl.com\":778},\"workers\":[{\"email\":\"mo1@brewhubphl.com\",\"regular\":799,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo11@brewhubphl.com\",\"regular\":799,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":7779}]},{\"id\":\"email_order_traps-014\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"mo1@brewhubphl.com\":601,\"mo11@brewhubphl.com\":602},\"workers\":[{\"email\":\"mo1@brewhubphl.com\",\"regular\":557,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo11@brewhubphl.com\",\"regular\":557,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":6017}]},{\"id\":\"email_order_traps-015\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"sam_k@brewhubphl.com\":265,\"sam.k@brewhubphl.com\":265,\"cy@brewhubphl.com\":529,\"cy47@brewhubphl.com\":530},\"workers\":[{\"email\":\"sam_k@brewhubphl.com\",\"regular\":570,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"sam.k@brewhubphl.com\",\"regular\":570,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy@brewhubphl.com\",\"regular\":1140,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy47@brewhubphl.com\",\"regular\":1140,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":7949}]},{\"id\":\"email_order_traps-016\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"jo@brewhubphl.com\":516,\"jo+pos@brewhubphl.com\":517},\"workers\":[{\"email\":\"jo@brewhubphl.com\",\"regular\":290,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo+pos@brewhubphl.com\",\"regular\":290,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":5167}]},{\"id\":\"email_order_traps-017\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"mo1@brewhubphl.com\":506,\"mo11@brewhubphl.com\":507},\"workers\":[{\"email\":\"mo1@brewhubphl.com\",\"regular\":730,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo11@brewhubphl.com\",\"regular\":730,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":5065}]},{\"id\":\"email_order_traps-018\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"b1@brewhubphl.com\":141,\"b10@brewhubphl.com\":142},\"workers\":[{\"email\":\"b1@brewhubphl.com\",\"regular\":597,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"b10@brewhubphl.com\",\"regular\":597,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1418}]},{\"id\":\"email_order_traps-019\",\"category\":\"email_order_traps\",\"tie_decides\":true,\"answer\":{\"b1@brewhubphl.com\":274,\"b10@brewhubphl.com\":275},\"workers\":[{\"email\":\"b1@brewhubphl.com\",\"regular\":64,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"b10@brewhubphl.com\",\"regular\":64,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":2748}]},{\"id\":\"zero_hours-000\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"cy@brewhubphl.com\":4563,\"idle5@brewhubphl.com\":0,\"ben@brewhubphl.com\":5848,\"ivo@brewhubphl.com\":5643,\"ana81@brewhubphl.com\":5649,\"cy89@brewhubphl.com\":408},\"workers\":[{\"email\":\"cy@brewhubphl.com\",\"regular\":1444,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle5@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":1851,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo@brewhubphl.com\",\"regular\":1786,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana81@brewhubphl.com\",\"regular\":1788,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy89@brewhubphl.com\",\"regular\":129,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":110557}]},{\"id\":\"zero_hours-001\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"idle4@brewhubphl.com\":0,\"kai57@brewhubphl.com\":1803,\"nia@brewhubphl.com\":1053,\"ana@brewhubphl.com\":463,\"fay@brewhubphl.com\":56,\"gus94@brewhubphl.com\":523,\"ivo@brewhubphl.com\":1987},\"workers\":[{\"email\":\"idle4@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai57@brewhubphl.com\",\"regular\":1922,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia@brewhubphl.com\",\"regular\":1122,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":493,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":60,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus94@brewhubphl.com\",\"regular\":558,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo@brewhubphl.com\",\"regular\":2118,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":29427}]},{\"id\":\"zero_hours-002\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"oz@brewhubphl.com\":7635,\"pia13@brewhubphl.com\":2734,\"fay@brewhubphl.com\":7758,\"idle6@brewhubphl.com\":0,\"kai84@brewhubphl.com\":6800,\"dee@brewhubphl.com\":6237,\"ana@brewhubphl.com\":169},\"workers\":[{\"email\":\"oz@brewhubphl.com\",\"regular\":2304,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia13@brewhubphl.com\",\"regular\":825,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":2341,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle6@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai84@brewhubphl.com\",\"regular\":2052,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":1882,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":51,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":156669}]},{\"id\":\"zero_hours-003\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"ana@brewhubphl.com\":4270,\"eli65@brewhubphl.com\":2717,\"cy22@brewhubphl.com\":607,\"idle9@brewhubphl.com\":0,\"hana6@brewhubphl.com\":4606},\"workers\":[{\"email\":\"ana@brewhubphl.com\",\"regular\":2060,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli65@brewhubphl.com\",\"regular\":1311,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy22@brewhubphl.com\",\"regular\":293,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle9@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana6@brewhubphl.com\",\"regular\":2222,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":61004}]},{\"id\":\"zero_hours-004\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"eli51@brewhubphl.com\":1643,\"idle1@brewhubphl.com\":0,\"jo@brewhubphl.com\":2640,\"gus@brewhubphl.com\":1309,\"fay@brewhubphl.com\":1867,\"hana@brewhubphl.com\":1128},\"workers\":[{\"email\":\"eli51@brewhubphl.com\",\"regular\":935,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle1@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":1503,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":745,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":1063,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":642,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":42936}]},{\"id\":\"zero_hours-005\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"idle3@brewhubphl.com\":0,\"fay74@brewhubphl.com\":18898,\"cy@brewhubphl.com\":5159,\"dee@brewhubphl.com\":18422},\"workers\":[{\"email\":\"idle3@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay74@brewhubphl.com\",\"regular\":1348,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy@brewhubphl.com\",\"regular\":368,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":1314,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":212398}]},{\"id\":\"zero_hours-006\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"nia32@brewhubphl.com\":3316,\"eli@brewhubphl.com\":4088,\"idle3@brewhubphl.com\":0,\"lu@brewhubphl.com\":4061,\"jo@brewhubphl.com\":1840,\"dee@brewhubphl.com\":2724,\"pia@brewhubphl.com\":135},\"workers\":[{\"email\":\"nia32@brewhubphl.com\",\"regular\":1826,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli@brewhubphl.com\",\"regular\":2251,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle3@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":2236,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":1013,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":1500,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":74,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":80820}]},{\"id\":\"zero_hours-007\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"lu1@brewhubphl.com\":9534,\"cy@brewhubphl.com\":3337,\"mo47@brewhubphl.com\":6989,\"lu@brewhubphl.com\":18726,\"ana6@brewhubphl.com\":8390,\"idle3@brewhubphl.com\":0},\"workers\":[{\"email\":\"lu1@brewhubphl.com\",\"regular\":1000,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy@brewhubphl.com\",\"regular\":350,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo47@brewhubphl.com\",\"regular\":733,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":1964,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana6@brewhubphl.com\",\"regular\":880,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle3@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":234882}]},{\"id\":\"zero_hours-008\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"jo16@brewhubphl.com\":6188,\"idle6@brewhubphl.com\":0,\"kai@brewhubphl.com\":5697,\"eli54@brewhubphl.com\":3483,\"mo38@brewhubphl.com\":7176,\"pia72@brewhubphl.com\":2695,\"eli@brewhubphl.com\":5429},\"workers\":[{\"email\":\"jo16@brewhubphl.com\",\"regular\":1917,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle6@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":1765,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli54@brewhubphl.com\",\"regular\":1079,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo38@brewhubphl.com\",\"regular\":2223,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia72@brewhubphl.com\",\"regular\":835,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli@brewhubphl.com\",\"regular\":1682,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":153342}]},{\"id\":\"zero_hours-009\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"kai1@brewhubphl.com\":9262,\"kai69@brewhubphl.com\":17317,\"jo@brewhubphl.com\":18645,\"idle8@brewhubphl.com\":0},\"workers\":[{\"email\":\"kai1@brewhubphl.com\",\"regular\":851,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai69@brewhubphl.com\",\"regular\":1591,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":1713,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle8@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":226120}]},{\"id\":\"zero_hours-010\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"idle3@brewhubphl.com\":0,\"kai@brewhubphl.com\":31,\"eli@brewhubphl.com\":488,\"jo@brewhubphl.com\":545,\"kai30@brewhubphl.com\":789},\"workers\":[{\"email\":\"idle3@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":89,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli@brewhubphl.com\",\"regular\":1408,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":1573,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai30@brewhubphl.com\",\"regular\":2274,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":9268}]},{\"id\":\"zero_hours-011\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"eli89@brewhubphl.com\":2302,\"idle5@brewhubphl.com\":0,\"ana31@brewhubphl.com\":1907,\"dee@brewhubphl.com\":1813},\"workers\":[{\"email\":\"eli89@brewhubphl.com\",\"regular\":1577,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle5@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana31@brewhubphl.com\",\"regular\":1307,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":1242,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":30113}]},{\"id\":\"zero_hours-012\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"jo20@brewhubphl.com\":13126,\"idle1@brewhubphl.com\":0,\"ben69@brewhubphl.com\":23244,\"jo@brewhubphl.com\":11826},\"workers\":[{\"email\":\"jo20@brewhubphl.com\",\"regular\":1222,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle1@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben69@brewhubphl.com\",\"regular\":2164,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":1101,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":240983}]},{\"id\":\"zero_hours-013\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"eli59@brewhubphl.com\":1842,\"idle9@brewhubphl.com\":0,\"ana52@brewhubphl.com\":14812,\"hana@brewhubphl.com\":3120,\"dee@brewhubphl.com\":10479,\"dee89@brewhubphl.com\":8236},\"workers\":[{\"email\":\"eli59@brewhubphl.com\",\"regular\":252,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle9@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana52@brewhubphl.com\",\"regular\":2027,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":427,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":1434,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee89@brewhubphl.com\",\"regular\":1127,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":192447}]},{\"id\":\"zero_hours-014\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"fay@brewhubphl.com\":10369,\"mo64@brewhubphl.com\":5987,\"idle8@brewhubphl.com\":0,\"ivo23@brewhubphl.com\":5049,\"lu@brewhubphl.com\":11862,\"mo@brewhubphl.com\":1011,\"kai@brewhubphl.com\":7876},\"workers\":[{\"email\":\"fay@brewhubphl.com\",\"regular\":2021,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo64@brewhubphl.com\",\"regular\":1167,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle8@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo23@brewhubphl.com\",\"regular\":984,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":2312,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":197,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":1535,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":210771}]},{\"id\":\"zero_hours-015\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"nia67@brewhubphl.com\":2244,\"ben@brewhubphl.com\":1908,\"lu@brewhubphl.com\":1878,\"idle8@brewhubphl.com\":0,\"oz@brewhubphl.com\":7972},\"workers\":[{\"email\":\"nia67@brewhubphl.com\",\"regular\":528,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":449,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":442,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle8@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":1876,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":70014}]},{\"id\":\"zero_hours-016\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"kai@brewhubphl.com\":11399,\"jo@brewhubphl.com\":13164,\"idle6@brewhubphl.com\":0,\"nia@brewhubphl.com\":15982},\"workers\":[{\"email\":\"kai@brewhubphl.com\",\"regular\":1634,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":1887,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle6@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia@brewhubphl.com\",\"regular\":2291,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":202728}]},{\"id\":\"zero_hours-017\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"idle9@brewhubphl.com\":0,\"ivo@brewhubphl.com\":8437,\"lu@brewhubphl.com\":7273,\"ana77@brewhubphl.com\":9108,\"dee@brewhubphl.com\":8905},\"workers\":[{\"email\":\"idle9@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo@brewhubphl.com\",\"regular\":2000,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":1724,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana77@brewhubphl.com\",\"regular\":2159,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":2111,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":168616}]},{\"id\":\"zero_hours-018\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"nia@brewhubphl.com\":3530,\"ana28@brewhubphl.com\":1758,\"idle3@brewhubphl.com\":0,\"ivo42@brewhubphl.com\":1503},\"workers\":[{\"email\":\"nia@brewhubphl.com\",\"regular\":1820,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana28@brewhubphl.com\",\"regular\":906,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"idle3@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo42@brewhubphl.com\",\"regular\":775,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":33956}]},{\"id\":\"zero_hours-019\",\"category\":\"zero_hours\",\"tie_decides\":false,\"answer\":{\"idle9@brewhubphl.com\":0,\"oz29@brewhubphl.com\":12260,\"hana71@brewhubphl.com\":24368,\"ben39@brewhubphl.com\":4145,\"eli25@brewhubphl.com\":6748},\"workers\":[{\"email\":\"idle9@brewhubphl.com\",\"regular\":0,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz29@brewhubphl.com\",\"regular\":565,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana71@brewhubphl.com\",\"regular\":1123,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben39@brewhubphl.com\",\"regular\":191,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli25@brewhubphl.com\",\"regular\":311,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":237605}]},{\"id\":\"dominant_worker-000\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"boss@brewhubphl.com\":8671,\"fay@brewhubphl.com\":337,\"mo@brewhubphl.com\":626},\"workers\":[{\"email\":\"boss@brewhubphl.com\",\"regular\":1647,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":64,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":119,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":48172}]},{\"id\":\"dominant_worker-001\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"ben@brewhubphl.com\":190,\"pia84@brewhubphl.com\":131,\"fay42@brewhubphl.com\":77,\"boss@brewhubphl.com\":7931,\"lu@brewhubphl.com\":59,\"ana@brewhubphl.com\":269,\"ivo10@brewhubphl.com\":155},\"workers\":[{\"email\":\"ben@brewhubphl.com\",\"regular\":77,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia84@brewhubphl.com\",\"regular\":53,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay42@brewhubphl.com\",\"regular\":31,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":3213,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":24,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":109,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo10@brewhubphl.com\",\"regular\":63,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":44062}]},{\"id\":\"dominant_worker-002\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"pia76@brewhubphl.com\":24,\"kai@brewhubphl.com\":233,\"dee@brewhubphl.com\":226,\"ana@brewhubphl.com\":202,\"boss@brewhubphl.com\":8050,\"cy5@brewhubphl.com\":209},\"workers\":[{\"email\":\"pia76@brewhubphl.com\",\"regular\":11,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":108,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":105,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":94,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":3735,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy5@brewhubphl.com\",\"regular\":97,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":44721}]},{\"id\":\"dominant_worker-003\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"hana85@brewhubphl.com\":139,\"boss@brewhubphl.com\":6559,\"hana@brewhubphl.com\":325,\"ana@brewhubphl.com\":126,\"gus@brewhubphl.com\":139},\"workers\":[{\"email\":\"hana85@brewhubphl.com\",\"regular\":32,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":1512,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":75,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":29,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":32,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":36444}]},{\"id\":\"dominant_worker-004\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"gus@brewhubphl.com\":860,\"hana@brewhubphl.com\":774,\"boss@brewhubphl.com\":14702},\"workers\":[{\"email\":\"gus@brewhubphl.com\",\"regular\":70,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":63,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":1197,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":81680}]},{\"id\":\"dominant_worker-005\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"hana80@brewhubphl.com\":147,\"lu@brewhubphl.com\":528,\"hana@brewhubphl.com\":605,\"boss@brewhubphl.com\":11517},\"workers\":[{\"email\":\"hana80@brewhubphl.com\",\"regular\":19,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":68,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":78,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":1485,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":63986}]},{\"id\":\"dominant_worker-006\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"pia@brewhubphl.com\":388,\"boss@brewhubphl.com\":5929,\"hana66@brewhubphl.com\":230,\"nia@brewhubphl.com\":41},\"workers\":[{\"email\":\"pia@brewhubphl.com\",\"regular\":96,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":1467,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana66@brewhubphl.com\",\"regular\":57,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia@brewhubphl.com\",\"regular\":10,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":32941}]},{\"id\":\"dominant_worker-007\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"hana@brewhubphl.com\":298,\"pia48@brewhubphl.com\":144,\"fay@brewhubphl.com\":51,\"cy62@brewhubphl.com\":196,\"dee@brewhubphl.com\":301,\"boss@brewhubphl.com\":10364,\"hana34@brewhubphl.com\":162},\"workers\":[{\"email\":\"hana@brewhubphl.com\",\"regular\":99,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia48@brewhubphl.com\",\"regular\":48,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":17,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy62@brewhubphl.com\",\"regular\":65,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":100,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":3447,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana34@brewhubphl.com\",\"regular\":54,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":57584}]},{\"id\":\"dominant_worker-008\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"ivo@brewhubphl.com\":213,\"gus1@brewhubphl.com\":573,\"gus@brewhubphl.com\":502,\"eli@brewhubphl.com\":260,\"boss@brewhubphl.com\":15944,\"fay@brewhubphl.com\":224},\"workers\":[{\"email\":\"ivo@brewhubphl.com\",\"regular\":36,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus1@brewhubphl.com\",\"regular\":97,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":85,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli@brewhubphl.com\",\"regular\":44,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":2700,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":38,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":88582}]},{\"id\":\"dominant_worker-009\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"gus@brewhubphl.com\":821,\"kai@brewhubphl.com\":821,\"boss@brewhubphl.com\":14777},\"workers\":[{\"email\":\"gus@brewhubphl.com\",\"regular\":19,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":19,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":342,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":82098}]},{\"id\":\"dominant_worker-010\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"jo@brewhubphl.com\":127,\"dee@brewhubphl.com\":110,\"boss@brewhubphl.com\":6880,\"jo97@brewhubphl.com\":136,\"gus@brewhubphl.com\":71,\"ben27@brewhubphl.com\":155,\"hana@brewhubphl.com\":165},\"workers\":[{\"email\":\"jo@brewhubphl.com\",\"regular\":80,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":69,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":4338,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo97@brewhubphl.com\",\"regular\":86,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":45,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben27@brewhubphl.com\",\"regular\":98,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":104,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":38223}]},{\"id\":\"dominant_worker-011\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"boss@brewhubphl.com\":17646,\"hana@brewhubphl.com\":395,\"eli@brewhubphl.com\":335,\"mo@brewhubphl.com\":319,\"nia32@brewhubphl.com\":380,\"ana@brewhubphl.com\":532},\"workers\":[{\"email\":\"boss@brewhubphl.com\",\"regular\":3483,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":78,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli@brewhubphl.com\",\"regular\":66,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":63,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia32@brewhubphl.com\",\"regular\":75,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":105,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":98035}]},{\"id\":\"dominant_worker-012\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"fay@brewhubphl.com\":149,\"boss@brewhubphl.com\":3887,\"fay18@brewhubphl.com\":137,\"pia93@brewhubphl.com\":146},\"workers\":[{\"email\":\"fay@brewhubphl.com\",\"regular\":91,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":2376,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay18@brewhubphl.com\",\"regular\":84,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia93@brewhubphl.com\",\"regular\":89,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":21598}]},{\"id\":\"dominant_worker-013\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"dee@brewhubphl.com\":933,\"boss@brewhubphl.com\":16964,\"lu88@brewhubphl.com\":952},\"workers\":[{\"email\":\"dee@brewhubphl.com\",\"regular\":94,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":1710,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu88@brewhubphl.com\",\"regular\":96,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":94247}]},{\"id\":\"dominant_worker-014\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"ben2@brewhubphl.com\":381,\"jo35@brewhubphl.com\":58,\"oz@brewhubphl.com\":176,\"boss@brewhubphl.com\":10651,\"pia@brewhubphl.com\":169,\"ana@brewhubphl.com\":32,\"eli86@brewhubphl.com\":368},\"workers\":[{\"email\":\"ben2@brewhubphl.com\",\"regular\":119,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo35@brewhubphl.com\",\"regular\":18,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":55,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":3330,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":53,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":10,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli86@brewhubphl.com\",\"regular\":115,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":59178}]},{\"id\":\"dominant_worker-015\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"kai1@brewhubphl.com\":58,\"hana@brewhubphl.com\":111,\"boss@brewhubphl.com\":2810,\"hana32@brewhubphl.com\":29,\"oz@brewhubphl.com\":114},\"workers\":[{\"email\":\"kai1@brewhubphl.com\",\"regular\":50,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":96,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":2421,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana32@brewhubphl.com\",\"regular\":25,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":98,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":15610}]},{\"id\":\"dominant_worker-016\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"ana@brewhubphl.com\":72,\"boss@brewhubphl.com\":5789,\"cy2@brewhubphl.com\":49,\"gus@brewhubphl.com\":137,\"fay52@brewhubphl.com\":206,\"lu13@brewhubphl.com\":65,\"ivo@brewhubphl.com\":114},\"workers\":[{\"email\":\"ana@brewhubphl.com\",\"regular\":41,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":3294,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy2@brewhubphl.com\",\"regular\":28,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":78,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay52@brewhubphl.com\",\"regular\":117,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu13@brewhubphl.com\",\"regular\":37,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo@brewhubphl.com\",\"regular\":65,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":32160}]},{\"id\":\"dominant_worker-017\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"nia@brewhubphl.com\":379,\"boss@brewhubphl.com\":15922,\"fay@brewhubphl.com\":446,\"mo99@brewhubphl.com\":312,\"ben@brewhubphl.com\":244,\"oz@brewhubphl.com\":388},\"workers\":[{\"email\":\"nia@brewhubphl.com\",\"regular\":79,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":3321,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":93,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo99@brewhubphl.com\",\"regular\":65,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":51,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":81,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":88455}]},{\"id\":\"dominant_worker-018\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"pia40@brewhubphl.com\":517,\"nia@brewhubphl.com\":1130,\"boss@brewhubphl.com\":14818},\"workers\":[{\"email\":\"pia40@brewhubphl.com\",\"regular\":54,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia@brewhubphl.com\",\"regular\":118,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":1548,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":82326}]},{\"id\":\"dominant_worker-019\",\"category\":\"dominant_worker\",\"tie_decides\":false,\"answer\":{\"ivo97@brewhubphl.com\":49,\"boss@brewhubphl.com\":3071,\"gus@brewhubphl.com\":108,\"ana@brewhubphl.com\":28,\"dee@brewhubphl.com\":100,\"mo3@brewhubphl.com\":56},\"workers\":[{\"email\":\"ivo97@brewhubphl.com\",\"regular\":40,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"boss@brewhubphl.com\",\"regular\":2484,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":87,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":23,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":81,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo3@brewhubphl.com\",\"regular\":45,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":17060}]},{\"id\":\"tiny_pool-000\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"eli18@brewhubphl.com\":1,\"ivo@brewhubphl.com\":1,\"nia@brewhubphl.com\":0,\"lu@brewhubphl.com\":0,\"kai@brewhubphl.com\":1,\"cy@brewhubphl.com\":0},\"workers\":[{\"email\":\"eli18@brewhubphl.com\",\"regular\":485,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo@brewhubphl.com\",\"regular\":341,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia@brewhubphl.com\",\"regular\":209,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":276,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":335,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy@brewhubphl.com\",\"regular\":169,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":17}]},{\"id\":\"tiny_pool-001\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"nia@brewhubphl.com\":1,\"ana@brewhubphl.com\":0,\"gus@brewhubphl.com\":0,\"fay@brewhubphl.com\":1,\"cy@brewhubphl.com\":0,\"jo79@brewhubphl.com\":1,\"ivo27@brewhubphl.com\":0,\"ivo5@brewhubphl.com\":1,\"lu@brewhubphl.com\":1},\"workers\":[{\"email\":\"nia@brewhubphl.com\",\"regular\":342,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":124,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":87,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":207,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy@brewhubphl.com\",\"regular\":181,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo79@brewhubphl.com\",\"regular\":223,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo27@brewhubphl.com\",\"regular\":178,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo5@brewhubphl.com\",\"regular\":195,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":220,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":25}]},{\"id\":\"tiny_pool-002\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"fay@brewhubphl.com\":1,\"ben@brewhubphl.com\":1,\"pia@brewhubphl.com\":0,\"jo84@brewhubphl.com\":0,\"ivo@brewhubphl.com\":1,\"jo@brewhubphl.com\":2,\"kai@brewhubphl.com\":1,\"eli25@brewhubphl.com\":1},\"workers\":[{\"email\":\"fay@brewhubphl.com\",\"regular\":198,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":464,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":153,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo84@brewhubphl.com\",\"regular\":141,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo@brewhubphl.com\",\"regular\":206,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":504,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":204,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli25@brewhubphl.com\",\"regular\":449,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":38}]},{\"id\":\"tiny_pool-003\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"oz37@brewhubphl.com\":0,\"ana@brewhubphl.com\":0,\"lu@brewhubphl.com\":0,\"kai78@brewhubphl.com\":0,\"ana43@brewhubphl.com\":0,\"jo@brewhubphl.com\":1,\"ivo6@brewhubphl.com\":1},\"workers\":[{\"email\":\"oz37@brewhubphl.com\",\"regular\":237,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":211,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":75,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai78@brewhubphl.com\",\"regular\":254,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana43@brewhubphl.com\",\"regular\":354,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":514,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo6@brewhubphl.com\",\"regular\":480,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":10}]},{\"id\":\"tiny_pool-004\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"lu24@brewhubphl.com\":1,\"nia@brewhubphl.com\":1,\"gus@brewhubphl.com\":0,\"cy44@brewhubphl.com\":0,\"pia33@brewhubphl.com\":1,\"mo@brewhubphl.com\":1},\"workers\":[{\"email\":\"lu24@brewhubphl.com\",\"regular\":556,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia@brewhubphl.com\",\"regular\":384,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":211,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy44@brewhubphl.com\",\"regular\":88,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia33@brewhubphl.com\",\"regular\":264,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":473,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":23}]},{\"id\":\"tiny_pool-005\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"oz@brewhubphl.com\":0,\"jo@brewhubphl.com\":1,\"pia23@brewhubphl.com\":1,\"kai39@brewhubphl.com\":1,\"nia89@brewhubphl.com\":0,\"dee@brewhubphl.com\":1,\"kai48@brewhubphl.com\":0,\"hana16@brewhubphl.com\":0},\"workers\":[{\"email\":\"oz@brewhubphl.com\",\"regular\":337,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":395,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia23@brewhubphl.com\",\"regular\":547,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai39@brewhubphl.com\",\"regular\":569,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia89@brewhubphl.com\",\"regular\":191,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":513,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai48@brewhubphl.com\",\"regular\":68,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana16@brewhubphl.com\",\"regular\":126,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":24}]},{\"id\":\"tiny_pool-006\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"hana@brewhubphl.com\":0,\"gus86@brewhubphl.com\":1,\"ivo@brewhubphl.com\":1,\"jo@brewhubphl.com\":0,\"gus52@brewhubphl.com\":0,\"gus@brewhubphl.com\":1,\"kai@brewhubphl.com\":0},\"workers\":[{\"email\":\"hana@brewhubphl.com\",\"regular\":445,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus86@brewhubphl.com\",\"regular\":490,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo@brewhubphl.com\",\"regular\":507,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":77,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus52@brewhubphl.com\",\"regular\":342,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":594,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":433,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":18}]},{\"id\":\"tiny_pool-007\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"eli51@brewhubphl.com\":0,\"pia13@brewhubphl.com\":1,\"jo69@brewhubphl.com\":0,\"oz@brewhubphl.com\":0,\"dee@brewhubphl.com\":1,\"lu@brewhubphl.com\":0},\"workers\":[{\"email\":\"eli51@brewhubphl.com\",\"regular\":456,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia13@brewhubphl.com\",\"regular\":535,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo69@brewhubphl.com\",\"regular\":272,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":298,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":489,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":406,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":11}]},{\"id\":\"tiny_pool-008\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"nia@brewhubphl.com\":0,\"jo15@brewhubphl.com\":0,\"jo95@brewhubphl.com\":0,\"ben@brewhubphl.com\":0,\"oz@brewhubphl.com\":1,\"pia@brewhubphl.com\":0},\"workers\":[{\"email\":\"nia@brewhubphl.com\",\"regular\":535,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo15@brewhubphl.com\",\"regular\":304,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo95@brewhubphl.com\",\"regular\":128,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":377,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":546,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":354,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":8}]},{\"id\":\"tiny_pool-009\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"hana@brewhubphl.com\":1,\"pia@brewhubphl.com\":0,\"ana@brewhubphl.com\":0,\"mo81@brewhubphl.com\":1,\"eli33@brewhubphl.com\":2,\"hana65@brewhubphl.com\":2,\"fay@brewhubphl.com\":0,\"kai@brewhubphl.com\":1,\"lu9@brewhubphl.com\":1},\"workers\":[{\"email\":\"hana@brewhubphl.com\",\"regular\":495,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":141,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":100,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo81@brewhubphl.com\",\"regular\":469,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli33@brewhubphl.com\",\"regular\":599,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana65@brewhubphl.com\",\"regular\":554,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":80,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":379,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu9@brewhubphl.com\",\"regular\":164,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":42}]},{\"id\":\"tiny_pool-010\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"jo84@brewhubphl.com\":0,\"oz@brewhubphl.com\":1,\"ben@brewhubphl.com\":0,\"lu@brewhubphl.com\":1,\"eli41@brewhubphl.com\":1,\"eli@brewhubphl.com\":0,\"fay@brewhubphl.com\":1},\"workers\":[{\"email\":\"jo84@brewhubphl.com\",\"regular\":106,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":565,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":411,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":478,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli41@brewhubphl.com\",\"regular\":526,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli@brewhubphl.com\",\"regular\":394,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":590,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":24}]},{\"id\":\"tiny_pool-011\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"nia11@brewhubphl.com\":1,\"cy37@brewhubphl.com\":0,\"pia@brewhubphl.com\":1,\"fay@brewhubphl.com\":0},\"workers\":[{\"email\":\"nia11@brewhubphl.com\",\"regular\":226,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy37@brewhubphl.com\",\"regular\":84,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":461,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":198,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":14}]},{\"id\":\"tiny_pool-012\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"dee@brewhubphl.com\":0,\"lu@brewhubphl.com\":1,\"ana@brewhubphl.com\":1,\"pia90@brewhubphl.com\":1,\"lu23@brewhubphl.com\":1,\"mo73@brewhubphl.com\":0,\"hana84@brewhubphl.com\":0},\"workers\":[{\"email\":\"dee@brewhubphl.com\",\"regular\":328,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":588,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":430,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia90@brewhubphl.com\",\"regular\":361,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu23@brewhubphl.com\",\"regular\":387,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo73@brewhubphl.com\",\"regular\":221,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana84@brewhubphl.com\",\"regular\":87,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":24}]},{\"id\":\"tiny_pool-013\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"lu@brewhubphl.com\":1,\"ben@brewhubphl.com\":1,\"oz@brewhubphl.com\":1,\"fay@brewhubphl.com\":1,\"nia27@brewhubphl.com\":0},\"workers\":[{\"email\":\"lu@brewhubphl.com\",\"regular\":299,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":363,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":292,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":311,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia27@brewhubphl.com\",\"regular\":234,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":22}]},{\"id\":\"tiny_pool-014\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"nia@brewhubphl.com\":0,\"eli71@brewhubphl.com\":1,\"ana@brewhubphl.com\":0,\"lu@brewhubphl.com\":0},\"workers\":[{\"email\":\"nia@brewhubphl.com\",\"regular\":534,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli71@brewhubphl.com\",\"regular\":581,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":130,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":251,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":7}]},{\"id\":\"tiny_pool-015\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"lu@brewhubphl.com\":0,\"gus@brewhubphl.com\":1,\"pia@brewhubphl.com\":0,\"ana61@brewhubphl.com\":0,\"mo@brewhubphl.com\":0,\"dee5@brewhubphl.com\":0,\"ivo5@brewhubphl.com\":1,\"eli@brewhubphl.com\":0,\"kai@brewhubphl.com\":1},\"workers\":[{\"email\":\"lu@brewhubphl.com\",\"regular\":101,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":546,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":394,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana61@brewhubphl.com\",\"regular\":515,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":109,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee5@brewhubphl.com\",\"regular\":444,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo5@brewhubphl.com\",\"regular\":566,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli@brewhubphl.com\",\"regular\":294,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":526,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":17}]},{\"id\":\"tiny_pool-016\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"jo44@brewhubphl.com\":0,\"hana@brewhubphl.com\":1,\"nia@brewhubphl.com\":1,\"fay67@brewhubphl.com\":0,\"fay61@brewhubphl.com\":1,\"nia77@brewhubphl.com\":0,\"ana12@brewhubphl.com\":1,\"fay@brewhubphl.com\":0,\"lu@brewhubphl.com\":1},\"workers\":[{\"email\":\"jo44@brewhubphl.com\",\"regular\":201,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":217,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia@brewhubphl.com\",\"regular\":357,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay67@brewhubphl.com\",\"regular\":160,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay61@brewhubphl.com\",\"regular\":254,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia77@brewhubphl.com\",\"regular\":144,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana12@brewhubphl.com\",\"regular\":395,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":137,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":400,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":28}]},{\"id\":\"tiny_pool-017\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"ana@brewhubphl.com\":1,\"pia77@brewhubphl.com\":1,\"dee21@brewhubphl.com\":1,\"nia92@brewhubphl.com\":1,\"kai73@brewhubphl.com\":0,\"kai@brewhubphl.com\":0},\"workers\":[{\"email\":\"ana@brewhubphl.com\",\"regular\":307,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia77@brewhubphl.com\",\"regular\":548,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee21@brewhubphl.com\",\"regular\":597,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia92@brewhubphl.com\",\"regular\":597,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai73@brewhubphl.com\",\"regular\":260,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":116,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":24}]},{\"id\":\"tiny_pool-018\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"oz@brewhubphl.com\":0,\"fay76@brewhubphl.com\":0,\"mo@brewhubphl.com\":1,\"pia@brewhubphl.com\":0},\"workers\":[{\"email\":\"oz@brewhubphl.com\",\"regular\":531,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay76@brewhubphl.com\",\"regular\":137,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":541,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":237,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":7}]},{\"id\":\"tiny_pool-019\",\"category\":\"tiny_pool\",\"tie_decides\":false,\"answer\":{\"hana@brewhubphl.com\":0,\"ana51@brewhubphl.com\":0,\"oz98@brewhubphl.com\":1,\"ivo65@brewhubphl.com\":1},\"workers\":[{\"email\":\"hana@brewhubphl.com\",\"regular\":120,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana51@brewhubphl.com\",\"regular\":135,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz98@brewhubphl.com\",\"regular\":222,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo65@brewhubphl.com\",\"regular\":447,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":11}]},{\"id\":\"many_workers-000\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"ben38@brewhubphl.com\":18243,\"oz@brewhubphl.com\":18433,\"ana46@brewhubphl.com\":6276,\"kai@brewhubphl.com\":9417,\"ben41@brewhubphl.com\":4714,\"mo67@brewhubphl.com\":8209,\"dee@brewhubphl.com\":22525,\"ana85@brewhubphl.com\":13154,\"fay4@brewhubphl.com\":10373,\"ivo@brewhubphl.com\":4143,\"gus15@brewhubphl.com\":17081,\"dee30@brewhubphl.com\":17755,\"lu@brewhubphl.com\":12059,\"nia@brewhubphl.com\":9422,\"fay32@brewhubphl.com\":23733,\"eli@brewhubphl.com\":11777},\"workers\":[{\"email\":\"ben38@brewhubphl.com\",\"regular\":547,\"overtime\":1976,\"sunday_regular\":600,\"sunday_overtime\":426},{\"email\":\"oz@brewhubphl.com\",\"regular\":1548,\"overtime\":437,\"sunday_regular\":322,\"sunday_overtime\":1279},{\"email\":\"ana46@brewhubphl.com\",\"regular\":486,\"overtime\":452,\"sunday_regular\":96,\"sunday_overtime\":187},{\"email\":\"kai@brewhubphl.com\",\"regular\":340,\"overtime\":504,\"sunday_regular\":921,\"sunday_overtime\":67},{\"email\":\"ben41@brewhubphl.com\",\"regular\":134,\"overtime\":517,\"sunday_regular\":178,\"sunday_overtime\":88},{\"email\":\"mo67@brewhubphl.com\",\"regular\":434,\"overtime\":982,\"sunday_regular\":164,\"sunday_overtime\":17},{\"email\":\"dee@brewhubphl.com\",\"regular\":61,\"overtime\":1755,\"sunday_regular\":1434,\"sunday_overtime\":1132},{\"email\":\"ana85@brewhubphl.com\",\"regular\":729,\"overtime\":1512,\"sunday_regular\":200,\"sunday_overtime\":118},{\"email\":\"fay4@brewhubphl.com\",\"regular\":133,\"overtime\":640,\"sunday_regular\":405,\"sunday_overtime\":840},{\"email\":\"ivo@brewhubphl.com\",\"regular\":77,\"overtime\":420,\"sunday_regular\":223,\"sunday_overtime\":86},{\"email\":\"gus15@brewhubphl.com\",\"regular\":908,\"overtime\":959,\"sunday_regular\":636,\"sunday_overtime\":820},{\"email\":\"dee30@brewhubphl.com\",\"regular\":1494,\"overtime\":392,\"sunday_regular\":1355,\"sunday_overtime\":213},{\"email\":\"lu@brewhubphl.com\",\"regular\":427,\"overtime\":278,\"sunday_regular\":1265,\"sunday_overtime\":376},{\"email\":\"nia@brewhubphl.com\",\"regular\":1576,\"overtime\":30,\"sunday_regular\":45,\"sunday_overtime\":182},{\"email\":\"fay32@brewhubphl.com\",\"regular\":3026,\"overtime\":874,\"sunday_regular\":435,\"sunday_overtime\":282},{\"email\":\"eli@brewhubphl.com\",\"regular\":1588,\"overtime\":184,\"sunday_regular\":56,\"sunday_overtime\":463}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1036572}]},{\"id\":\"many_workers-001\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"mo@brewhubphl.com\":2024,\"gus@brewhubphl.com\":4126,\"nia52@brewhubphl.com\":389,\"kai42@brewhubphl.com\":1568,\"fay@brewhubphl.com\":4719,\"cy63@brewhubphl.com\":2717,\"cy@brewhubphl.com\":4653,\"ben@brewhubphl.com\":952,\"fay40@brewhubphl.com\":4697,\"dee@brewhubphl.com\":1596,\"hana@brewhubphl.com\":4515,\"lu@brewhubphl.com\":1571,\"gus86@brewhubphl.com\":4672,\"pia16@brewhubphl.com\":867},\"workers\":[{\"email\":\"mo@brewhubphl.com\",\"regular\":542,\"overtime\":1447,\"sunday_regular\":29,\"sunday_overtime\":25},{\"email\":\"gus@brewhubphl.com\",\"regular\":2602,\"overtime\":523,\"sunday_regular\":238,\"sunday_overtime\":802},{\"email\":\"nia52@brewhubphl.com\",\"regular\":57,\"overtime\":177,\"sunday_regular\":141,\"sunday_overtime\":18},{\"email\":\"kai42@brewhubphl.com\",\"regular\":13,\"overtime\":440,\"sunday_regular\":638,\"sunday_overtime\":492},{\"email\":\"fay@brewhubphl.com\",\"regular\":978,\"overtime\":1142,\"sunday_regular\":1331,\"sunday_overtime\":1313},{\"email\":\"cy63@brewhubphl.com\",\"regular\":871,\"overtime\":1069,\"sunday_regular\":471,\"sunday_overtime\":332},{\"email\":\"cy@brewhubphl.com\",\"regular\":2615,\"overtime\":1531,\"sunday_regular\":239,\"sunday_overtime\":312},{\"email\":\"ben@brewhubphl.com\",\"regular\":710,\"overtime\":200,\"sunday_regular\":36,\"sunday_overtime\":15},{\"email\":\"fay40@brewhubphl.com\",\"regular\":2147,\"overtime\":1175,\"sunday_regular\":208,\"sunday_overtime\":1211},{\"email\":\"dee@brewhubphl.com\",\"regular\":73,\"overtime\":358,\"sunday_regular\":1140,\"sunday_overtime\":40},{\"email\":\"hana@brewhubphl.com\",\"regular\":4233,\"overtime\":77,\"sunday_regular\":106,\"sunday_overtime\":142},{\"email\":\"lu@brewhubphl.com\",\"regular\":1316,\"overtime\":155,\"sunday_regular\":103,\"sunday_overtime\":12},{\"email\":\"gus86@brewhubphl.com\",\"regular\":3744,\"overtime\":619,\"sunday_regular\":262,\"sunday_overtime\":91},{\"email\":\"pia16@brewhubphl.com\",\"regular\":350,\"overtime\":12,\"sunday_regular\":466,\"sunday_overtime\":47}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":195334}]},{\"id\":\"many_workers-002\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"pia50@brewhubphl.com\":2034,\"jo@brewhubphl.com\":6486,\"ivo55@brewhubphl.com\":2086,\"cy@brewhubphl.com\":3257,\"hana74@brewhubphl.com\":2835,\"kai@brewhubphl.com\":2640,\"fay47@brewhubphl.com\":1539,\"ben@brewhubphl.com\":3155,\"fay@brewhubphl.com\":1923,\"ana@brewhubphl.com\":3420,\"fay35@brewhubphl.com\":6990,\"dee1@brewhubphl.com\":6101,\"ana93@brewhubphl.com\":958,\"pia@brewhubphl.com\":6956},\"workers\":[{\"email\":\"pia50@brewhubphl.com\",\"regular\":628,\"overtime\":211,\"sunday_regular\":84,\"sunday_overtime\":462},{\"email\":\"jo@brewhubphl.com\",\"regular\":901,\"overtime\":3506,\"sunday_regular\":0,\"sunday_overtime\":8},{\"email\":\"ivo55@brewhubphl.com\",\"regular\":1049,\"overtime\":190,\"sunday_regular\":180,\"sunday_overtime\":1},{\"email\":\"cy@brewhubphl.com\",\"regular\":213,\"overtime\":660,\"sunday_regular\":541,\"sunday_overtime\":803},{\"email\":\"hana74@brewhubphl.com\",\"regular\":1015,\"overtime\":826,\"sunday_regular\":77,\"sunday_overtime\":12},{\"email\":\"kai@brewhubphl.com\",\"regular\":1035,\"overtime\":235,\"sunday_regular\":497,\"sunday_overtime\":30},{\"email\":\"fay47@brewhubphl.com\",\"regular\":59,\"overtime\":695,\"sunday_regular\":110,\"sunday_overtime\":184},{\"email\":\"ben@brewhubphl.com\",\"regular\":1253,\"overtime\":22,\"sunday_regular\":308,\"sunday_overtime\":565},{\"email\":\"fay@brewhubphl.com\",\"regular\":1071,\"overtime\":191,\"sunday_regular\":28,\"sunday_overtime\":19},{\"email\":\"ana@brewhubphl.com\",\"regular\":1605,\"overtime\":403,\"sunday_regular\":215,\"sunday_overtime\":105},{\"email\":\"fay35@brewhubphl.com\",\"regular\":799,\"overtime\":3734,\"sunday_regular\":78,\"sunday_overtime\":147},{\"email\":\"dee1@brewhubphl.com\",\"regular\":1216,\"overtime\":624,\"sunday_regular\":674,\"sunday_overtime\":1639},{\"email\":\"ana93@brewhubphl.com\",\"regular\":317,\"overtime\":78,\"sunday_regular\":68,\"sunday_overtime\":189},{\"email\":\"pia@brewhubphl.com\",\"regular\":349,\"overtime\":1841,\"sunday_regular\":2101,\"sunday_overtime\":444}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":251901}]},{\"id\":\"many_workers-003\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"fay@brewhubphl.com\":1917,\"ivo@brewhubphl.com\":29675,\"pia@brewhubphl.com\":21762,\"gus30@brewhubphl.com\":11981,\"gus@brewhubphl.com\":51536,\"dee55@brewhubphl.com\":6628,\"oz46@brewhubphl.com\":46479,\"oz9@brewhubphl.com\":47926,\"hana@brewhubphl.com\":13354,\"ben16@brewhubphl.com\":55666,\"eli@brewhubphl.com\":3264,\"nia11@brewhubphl.com\":29750,\"jo77@brewhubphl.com\":46578},\"workers\":[{\"email\":\"fay@brewhubphl.com\",\"regular\":41,\"overtime\":101,\"sunday_regular\":13,\"sunday_overtime\":0},{\"email\":\"ivo@brewhubphl.com\",\"regular\":688,\"overtime\":418,\"sunday_regular\":1205,\"sunday_overtime\":89},{\"email\":\"pia@brewhubphl.com\",\"regular\":477,\"overtime\":543,\"sunday_regular\":629,\"sunday_overtime\":111},{\"email\":\"gus30@brewhubphl.com\",\"regular\":761,\"overtime\":40,\"sunday_regular\":13,\"sunday_overtime\":155},{\"email\":\"gus@brewhubphl.com\",\"regular\":2621,\"overtime\":1435,\"sunday_regular\":1,\"sunday_overtime\":111},{\"email\":\"dee55@brewhubphl.com\",\"regular\":501,\"overtime\":3,\"sunday_regular\":11,\"sunday_overtime\":21},{\"email\":\"oz46@brewhubphl.com\",\"regular\":1673,\"overtime\":631,\"sunday_regular\":300,\"sunday_overtime\":1155},{\"email\":\"oz9@brewhubphl.com\",\"regular\":2677,\"overtime\":21,\"sunday_regular\":12,\"sunday_overtime\":1166},{\"email\":\"hana@brewhubphl.com\",\"regular\":1037,\"overtime\":31,\"sunday_regular\":5,\"sunday_overtime\":7},{\"email\":\"ben16@brewhubphl.com\",\"regular\":1105,\"overtime\":833,\"sunday_regular\":2460,\"sunday_overtime\":104},{\"email\":\"eli@brewhubphl.com\",\"regular\":12,\"overtime\":241,\"sunday_regular\":4,\"sunday_overtime\":7},{\"email\":\"nia11@brewhubphl.com\",\"regular\":2223,\"overtime\":51,\"sunday_regular\":25,\"sunday_overtime\":107},{\"email\":\"jo77@brewhubphl.com\",\"regular\":2641,\"overtime\":719,\"sunday_regular\":383,\"sunday_overtime\":24}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1832584}]},{\"id\":\"many_workers-004\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"ivo57@brewhubphl.com\":28101,\"jo@brewhubphl.com\":21756,\"cy@brewhubphl.com\":29701,\"cy13@brewhubphl.com\":9336,\"gus4@brewhubphl.com\":24892,\"ben@brewhubphl.com\":29919,\"kai@brewhubphl.com\":11928,\"ana@brewhubphl.com\":22177,\"oz30@brewhubphl.com\":14637,\"ivo@brewhubphl.com\":20228,\"cy78@brewhubphl.com\":1137,\"hana@brewhubphl.com\":4107,\"hana2@brewhubphl.com\":9734,\"fay@brewhubphl.com\":13898},\"workers\":[{\"email\":\"ivo57@brewhubphl.com\",\"regular\":1957,\"overtime\":1818,\"sunday_regular\":105,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":385,\"overtime\":981,\"sunday_regular\":421,\"sunday_overtime\":1217},{\"email\":\"cy@brewhubphl.com\",\"regular\":3850,\"overtime\":91,\"sunday_regular\":81,\"sunday_overtime\":79},{\"email\":\"cy13@brewhubphl.com\",\"regular\":202,\"overtime\":447,\"sunday_regular\":131,\"sunday_overtime\":509},{\"email\":\"gus4@brewhubphl.com\",\"regular\":68,\"overtime\":2907,\"sunday_regular\":199,\"sunday_overtime\":263},{\"email\":\"ben@brewhubphl.com\",\"regular\":2817,\"overtime\":1000,\"sunday_regular\":172,\"sunday_overtime\":142},{\"email\":\"kai@brewhubphl.com\",\"regular\":293,\"overtime\":1344,\"sunday_regular\":5,\"sunday_overtime\":5},{\"email\":\"ana@brewhubphl.com\",\"regular\":1808,\"overtime\":556,\"sunday_regular\":595,\"sunday_overtime\":103},{\"email\":\"oz30@brewhubphl.com\",\"regular\":1834,\"overtime\":126,\"sunday_regular\":23,\"sunday_overtime\":38},{\"email\":\"ivo@brewhubphl.com\",\"regular\":1437,\"overtime\":668,\"sunday_regular\":248,\"sunday_overtime\":440},{\"email\":\"cy78@brewhubphl.com\",\"regular\":113,\"overtime\":31,\"sunday_regular\":0,\"sunday_overtime\":13},{\"email\":\"hana@brewhubphl.com\",\"regular\":510,\"overtime\":47,\"sunday_regular\":2,\"sunday_overtime\":8},{\"email\":\"hana2@brewhubphl.com\",\"regular\":1284,\"overtime\":31,\"sunday_regular\":2,\"sunday_overtime\":27},{\"email\":\"fay@brewhubphl.com\",\"regular\":1824,\"overtime\":37,\"sunday_regular\":24,\"sunday_overtime\":34}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1207759}]},{\"id\":\"many_workers-005\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"gus@brewhubphl.com\":3998,\"kai73@brewhubphl.com\":12441,\"fay94@brewhubphl.com\":514,\"lu@brewhubphl.com\":4138,\"ana8@brewhubphl.com\":13676,\"fay9@brewhubphl.com\":11183,\"jo@brewhubphl.com\":3980,\"ana@brewhubphl.com\":12064,\"kai@brewhubphl.com\":3402,\"cy@brewhubphl.com\":13058,\"dee@brewhubphl.com\":112,\"ana96@brewhubphl.com\":5081,\"mo@brewhubphl.com\":11189},\"workers\":[{\"email\":\"gus@brewhubphl.com\",\"regular\":1265,\"overtime\":19,\"sunday_regular\":26,\"sunday_overtime\":5},{\"email\":\"kai73@brewhubphl.com\",\"regular\":192,\"overtime\":3290,\"sunday_regular\":279,\"sunday_overtime\":331},{\"email\":\"fay94@brewhubphl.com\",\"regular\":114,\"overtime\":27,\"sunday_regular\":12,\"sunday_overtime\":16},{\"email\":\"lu@brewhubphl.com\",\"regular\":649,\"overtime\":342,\"sunday_regular\":282,\"sunday_overtime\":88},{\"email\":\"ana8@brewhubphl.com\",\"regular\":3515,\"overtime\":859,\"sunday_regular\":118,\"sunday_overtime\":6},{\"email\":\"fay9@brewhubphl.com\",\"regular\":2828,\"overtime\":31,\"sunday_regular\":565,\"sunday_overtime\":254},{\"email\":\"jo@brewhubphl.com\",\"regular\":12,\"overtime\":140,\"sunday_regular\":890,\"sunday_overtime\":267},{\"email\":\"ana@brewhubphl.com\",\"regular\":991,\"overtime\":2060,\"sunday_regular\":106,\"sunday_overtime\":811},{\"email\":\"kai@brewhubphl.com\",\"regular\":999,\"overtime\":56,\"sunday_regular\":29,\"sunday_overtime\":35},{\"email\":\"cy@brewhubphl.com\",\"regular\":393,\"overtime\":1179,\"sunday_regular\":1958,\"sunday_overtime\":765},{\"email\":\"dee@brewhubphl.com\",\"regular\":2,\"overtime\":6,\"sunday_regular\":6,\"sunday_overtime\":23},{\"email\":\"ana96@brewhubphl.com\",\"regular\":1649,\"overtime\":15,\"sunday_regular\":3,\"sunday_overtime\":4},{\"email\":\"mo@brewhubphl.com\",\"regular\":3187,\"overtime\":484,\"sunday_regular\":7,\"sunday_overtime\":2}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":474183}]},{\"id\":\"many_workers-006\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"lu2@brewhubphl.com\":20066,\"oz25@brewhubphl.com\":42183,\"nia@brewhubphl.com\":6189,\"kai54@brewhubphl.com\":1204,\"lu@brewhubphl.com\":46219,\"kai49@brewhubphl.com\":15889,\"jo43@brewhubphl.com\":47129,\"pia84@brewhubphl.com\":60942,\"gus@brewhubphl.com\":35110,\"dee82@brewhubphl.com\":42273,\"cy6@brewhubphl.com\":23834,\"fay@brewhubphl.com\":13211},\"workers\":[{\"email\":\"lu2@brewhubphl.com\",\"regular\":54,\"overtime\":567,\"sunday_regular\":446,\"sunday_overtime\":499},{\"email\":\"oz25@brewhubphl.com\",\"regular\":2867,\"overtime\":67,\"sunday_regular\":305,\"sunday_overtime\":53},{\"email\":\"nia@brewhubphl.com\",\"regular\":73,\"overtime\":363,\"sunday_regular\":34,\"sunday_overtime\":13},{\"email\":\"kai54@brewhubphl.com\",\"regular\":65,\"overtime\":15,\"sunday_regular\":0,\"sunday_overtime\":14},{\"email\":\"lu@brewhubphl.com\",\"regular\":3452,\"overtime\":68,\"sunday_regular\":10,\"sunday_overtime\":77},{\"email\":\"kai49@brewhubphl.com\",\"regular\":26,\"overtime\":963,\"sunday_regular\":236,\"sunday_overtime\":15},{\"email\":\"jo43@brewhubphl.com\",\"regular\":1077,\"overtime\":314,\"sunday_regular\":1274,\"sunday_overtime\":1013},{\"email\":\"pia84@brewhubphl.com\",\"regular\":464,\"overtime\":1506,\"sunday_regular\":2250,\"sunday_overtime\":536},{\"email\":\"gus@brewhubphl.com\",\"regular\":1509,\"overtime\":840,\"sunday_regular\":277,\"sunday_overtime\":114},{\"email\":\"dee82@brewhubphl.com\",\"regular\":1210,\"overtime\":1501,\"sunday_regular\":195,\"sunday_overtime\":393},{\"email\":\"cy6@brewhubphl.com\",\"regular\":1254,\"overtime\":160,\"sunday_regular\":84,\"sunday_overtime\":362},{\"email\":\"fay@brewhubphl.com\",\"regular\":453,\"overtime\":350,\"sunday_regular\":140,\"sunday_overtime\":88}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1771248}]},{\"id\":\"many_workers-007\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"dee@brewhubphl.com\":5492,\"fay86@brewhubphl.com\":13082,\"gus57@brewhubphl.com\":8093,\"ben@brewhubphl.com\":12808,\"mo@brewhubphl.com\":7150,\"gus@brewhubphl.com\":18188,\"ana@brewhubphl.com\":204,\"pia49@brewhubphl.com\":4250,\"kai@brewhubphl.com\":8039,\"hana@brewhubphl.com\":4911,\"jo@brewhubphl.com\":5596,\"fay@brewhubphl.com\":5555,\"oz56@brewhubphl.com\":9551},\"workers\":[{\"email\":\"dee@brewhubphl.com\",\"regular\":827,\"overtime\":152,\"sunday_regular\":26,\"sunday_overtime\":317},{\"email\":\"fay86@brewhubphl.com\",\"regular\":1918,\"overtime\":962,\"sunday_regular\":155,\"sunday_overtime\":114},{\"email\":\"gus57@brewhubphl.com\",\"regular\":602,\"overtime\":938,\"sunday_regular\":325,\"sunday_overtime\":83},{\"email\":\"ben@brewhubphl.com\",\"regular\":1013,\"overtime\":1534,\"sunday_regular\":133,\"sunday_overtime\":403},{\"email\":\"mo@brewhubphl.com\",\"regular\":740,\"overtime\":156,\"sunday_regular\":361,\"sunday_overtime\":464},{\"email\":\"gus@brewhubphl.com\",\"regular\":2193,\"overtime\":457,\"sunday_regular\":1556,\"sunday_overtime\":172},{\"email\":\"ana@brewhubphl.com\",\"regular\":33,\"overtime\":0,\"sunday_regular\":13,\"sunday_overtime\":3},{\"email\":\"pia49@brewhubphl.com\",\"regular\":909,\"overtime\":97,\"sunday_regular\":10,\"sunday_overtime\":7},{\"email\":\"kai@brewhubphl.com\",\"regular\":631,\"overtime\":1085,\"sunday_regular\":82,\"sunday_overtime\":137},{\"email\":\"hana@brewhubphl.com\",\"regular\":512,\"overtime\":388,\"sunday_regular\":66,\"sunday_overtime\":216},{\"email\":\"jo@brewhubphl.com\",\"regular\":820,\"overtime\":335,\"sunday_regular\":115,\"sunday_overtime\":77},{\"email\":\"fay@brewhubphl.com\",\"regular\":444,\"overtime\":475,\"sunday_regular\":287,\"sunday_overtime\":131},{\"email\":\"oz56@brewhubphl.com\",\"regular\":2151,\"overtime\":13,\"sunday_regular\":55,\"sunday_overtime\":80}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":514595}]},{\"id\":\"many_workers-008\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"eli@brewhubphl.com\":640,\"cy7@brewhubphl.com\":521,\"fay@brewhubphl.com\":2552,\"lu60@brewhubphl.com\":1428,\"pia@brewhubphl.com\":1385,\"oz@brewhubphl.com\":1512,\"ben@brewhubphl.com\":1166,\"dee@brewhubphl.com\":82,\"oz26@brewhubphl.com\":1983,\"ivo@brewhubphl.com\":2120,\"hana@brewhubphl.com\":270,\"lu30@brewhubphl.com\":631,\"ivo87@brewhubphl.com\":2403},\"workers\":[{\"email\":\"eli@brewhubphl.com\",\"regular\":175,\"overtime\":405,\"sunday_regular\":4,\"sunday_overtime\":570},{\"email\":\"cy7@brewhubphl.com\",\"regular\":183,\"overtime\":677,\"sunday_regular\":8,\"sunday_overtime\":70},{\"email\":\"fay@brewhubphl.com\",\"regular\":817,\"overtime\":1724,\"sunday_regular\":215,\"sunday_overtime\":1842},{\"email\":\"lu60@brewhubphl.com\",\"regular\":321,\"overtime\":160,\"sunday_regular\":1118,\"sunday_overtime\":974},{\"email\":\"pia@brewhubphl.com\",\"regular\":2084,\"overtime\":108,\"sunday_regular\":235,\"sunday_overtime\":69},{\"email\":\"oz@brewhubphl.com\",\"regular\":477,\"overtime\":1359,\"sunday_regular\":234,\"sunday_overtime\":654},{\"email\":\"ben@brewhubphl.com\",\"regular\":234,\"overtime\":676,\"sunday_regular\":592,\"sunday_overtime\":599},{\"email\":\"dee@brewhubphl.com\",\"regular\":102,\"overtime\":18,\"sunday_regular\":13,\"sunday_overtime\":15},{\"email\":\"oz26@brewhubphl.com\",\"regular\":381,\"overtime\":1213,\"sunday_regular\":65,\"sunday_overtime\":1915},{\"email\":\"ivo@brewhubphl.com\",\"regular\":3035,\"overtime\":610,\"sunday_regular\":3,\"sunday_overtime\":172},{\"email\":\"hana@brewhubphl.com\",\"regular\":313,\"overtime\":61,\"sunday_regular\":110,\"sunday_overtime\":2},{\"email\":\"lu30@brewhubphl.com\",\"regular\":941,\"overtime\":151,\"sunday_regular\":36,\"sunday_overtime\":9},{\"email\":\"ivo87@brewhubphl.com\",\"regular\":2117,\"overtime\":1757,\"sunday_regular\":151,\"sunday_overtime\":306}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":83468}]},{\"id\":\"many_workers-009\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"hana5@brewhubphl.com\":23734,\"lu19@brewhubphl.com\":29955,\"gus@brewhubphl.com\":11771,\"nia@brewhubphl.com\":31555,\"ben@brewhubphl.com\":32491,\"kai@brewhubphl.com\":10249,\"oz@brewhubphl.com\":19785,\"dee75@brewhubphl.com\":19613,\"ivo@brewhubphl.com\":7092,\"dee@brewhubphl.com\":1564,\"cy92@brewhubphl.com\":9542,\"eli@brewhubphl.com\":18885,\"kai94@brewhubphl.com\":29148},\"workers\":[{\"email\":\"hana5@brewhubphl.com\",\"regular\":1460,\"overtime\":1498,\"sunday_regular\":352,\"sunday_overtime\":13},{\"email\":\"lu19@brewhubphl.com\",\"regular\":2741,\"overtime\":699,\"sunday_regular\":264,\"sunday_overtime\":490},{\"email\":\"gus@brewhubphl.com\",\"regular\":972,\"overtime\":212,\"sunday_regular\":460,\"sunday_overtime\":4},{\"email\":\"nia@brewhubphl.com\",\"regular\":2123,\"overtime\":1822,\"sunday_regular\":2,\"sunday_overtime\":471},{\"email\":\"ben@brewhubphl.com\",\"regular\":4331,\"overtime\":47,\"sunday_regular\":37,\"sunday_overtime\":134},{\"email\":\"kai@brewhubphl.com\",\"regular\":995,\"overtime\":250,\"sunday_regular\":147,\"sunday_overtime\":43},{\"email\":\"oz@brewhubphl.com\",\"regular\":2539,\"overtime\":187,\"sunday_regular\":42,\"sunday_overtime\":2},{\"email\":\"dee75@brewhubphl.com\",\"regular\":914,\"overtime\":1515,\"sunday_regular\":119,\"sunday_overtime\":198},{\"email\":\"ivo@brewhubphl.com\",\"regular\":343,\"overtime\":449,\"sunday_regular\":34,\"sunday_overtime\":167},{\"email\":\"dee@brewhubphl.com\",\"regular\":195,\"overtime\":21,\"sunday_regular\":2,\"sunday_overtime\":1},{\"email\":\"cy92@brewhubphl.com\",\"regular\":1038,\"overtime\":126,\"sunday_regular\":82,\"sunday_overtime\":90},{\"email\":\"eli@brewhubphl.com\",\"regular\":1900,\"overtime\":265,\"sunday_regular\":293,\"sunday_overtime\":186},{\"email\":\"kai94@brewhubphl.com\",\"regular\":400,\"overtime\":1522,\"sunday_regular\":719,\"sunday_overtime\":1440}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1226920}]},{\"id\":\"many_workers-010\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"jo@brewhubphl.com\":29957,\"lu@brewhubphl.com\":20600,\"ivo79@brewhubphl.com\":1710,\"ivo@brewhubphl.com\":4517,\"ben@brewhubphl.com\":22625,\"fay@brewhubphl.com\":19849,\"kai37@brewhubphl.com\":18083,\"pia@brewhubphl.com\":8303,\"hana@brewhubphl.com\":8297,\"eli@brewhubphl.com\":27887,\"dee29@brewhubphl.com\":20064,\"oz@brewhubphl.com\":25338,\"mo23@brewhubphl.com\":9003,\"dee@brewhubphl.com\":15641,\"hana47@brewhubphl.com\":10492},\"workers\":[{\"email\":\"jo@brewhubphl.com\",\"regular\":1796,\"overtime\":1807,\"sunday_regular\":79,\"sunday_overtime\":1066},{\"email\":\"lu@brewhubphl.com\",\"regular\":1798,\"overtime\":795,\"sunday_regular\":365,\"sunday_overtime\":307},{\"email\":\"ivo79@brewhubphl.com\",\"regular\":8,\"overtime\":161,\"sunday_regular\":8,\"sunday_overtime\":94},{\"email\":\"ivo@brewhubphl.com\",\"regular\":445,\"overtime\":102,\"sunday_regular\":77,\"sunday_overtime\":92},{\"email\":\"ben@brewhubphl.com\",\"regular\":95,\"overtime\":1664,\"sunday_regular\":1053,\"sunday_overtime\":774},{\"email\":\"fay@brewhubphl.com\",\"regular\":236,\"overtime\":2284,\"sunday_regular\":503,\"sunday_overtime\":123},{\"email\":\"kai37@brewhubphl.com\",\"regular\":524,\"overtime\":305,\"sunday_regular\":1639,\"sunday_overtime\":398},{\"email\":\"pia@brewhubphl.com\",\"regular\":556,\"overtime\":244,\"sunday_regular\":162,\"sunday_overtime\":354},{\"email\":\"hana@brewhubphl.com\",\"regular\":337,\"overtime\":130,\"sunday_regular\":461,\"sunday_overtime\":387},{\"email\":\"eli@brewhubphl.com\",\"regular\":774,\"overtime\":167,\"sunday_regular\":1748,\"sunday_overtime\":1731},{\"email\":\"dee29@brewhubphl.com\",\"regular\":1640,\"overtime\":1350,\"sunday_regular\":17,\"sunday_overtime\":173},{\"email\":\"oz@brewhubphl.com\",\"regular\":3603,\"overtime\":190,\"sunday_regular\":162,\"sunday_overtime\":61},{\"email\":\"mo23@brewhubphl.com\",\"regular\":1010,\"overtime\":338,\"sunday_regular\":79,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":1123,\"overtime\":203,\"sunday_regular\":802,\"sunday_overtime\":351},{\"email\":\"hana47@brewhubphl.com\",\"regular\":1321,\"overtime\":221,\"sunday_regular\":88,\"sunday_overtime\":33}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1211830}]},{\"id\":\"many_workers-011\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"mo36@brewhubphl.com\":16270,\"nia@brewhubphl.com\":25101,\"hana@brewhubphl.com\":13935,\"ana1@brewhubphl.com\":14494,\"dee70@brewhubphl.com\":13801,\"ivo@brewhubphl.com\":17396,\"oz@brewhubphl.com\":13201,\"nia59@brewhubphl.com\":26760,\"eli@brewhubphl.com\":39369,\"dee@brewhubphl.com\":27019,\"fay@brewhubphl.com\":18997,\"ivo21@brewhubphl.com\":14827,\"dee29@brewhubphl.com\":28061,\"gus@brewhubphl.com\":32973,\"ana@brewhubphl.com\":2343,\"oz23@brewhubphl.com\":11917},\"workers\":[{\"email\":\"mo36@brewhubphl.com\",\"regular\":632,\"overtime\":767,\"sunday_regular\":495,\"sunday_overtime\":57},{\"email\":\"nia@brewhubphl.com\",\"regular\":1035,\"overtime\":1378,\"sunday_regular\":130,\"sunday_overtime\":467},{\"email\":\"hana@brewhubphl.com\",\"regular\":436,\"overtime\":978,\"sunday_regular\":194,\"sunday_overtime\":63},{\"email\":\"ana1@brewhubphl.com\",\"regular\":465,\"overtime\":1197,\"sunday_regular\":55,\"sunday_overtime\":21},{\"email\":\"dee70@brewhubphl.com\",\"regular\":749,\"overtime\":620,\"sunday_regular\":37,\"sunday_overtime\":249},{\"email\":\"ivo@brewhubphl.com\",\"regular\":1532,\"overtime\":237,\"sunday_regular\":68,\"sunday_overtime\":249},{\"email\":\"oz@brewhubphl.com\",\"regular\":269,\"overtime\":633,\"sunday_regular\":284,\"sunday_overtime\":397},{\"email\":\"nia59@brewhubphl.com\",\"regular\":479,\"overtime\":1616,\"sunday_regular\":520,\"sunday_overtime\":594},{\"email\":\"eli@brewhubphl.com\",\"regular\":1484,\"overtime\":2600,\"sunday_regular\":184,\"sunday_overtime\":453},{\"email\":\"dee@brewhubphl.com\",\"regular\":919,\"overtime\":408,\"sunday_regular\":1757,\"sunday_overtime\":156},{\"email\":\"fay@brewhubphl.com\",\"regular\":2151,\"overtime\":37,\"sunday_regular\":60,\"sunday_overtime\":30},{\"email\":\"ivo21@brewhubphl.com\",\"regular\":902,\"overtime\":655,\"sunday_regular\":176,\"sunday_overtime\":45},{\"email\":\"dee29@brewhubphl.com\",\"regular\":2159,\"overtime\":234,\"sunday_regular\":429,\"sunday_overtime\":543},{\"email\":\"gus@brewhubphl.com\",\"regular\":756,\"overtime\":191,\"sunday_regular\":1845,\"sunday_overtime\":1162},{\"email\":\"ana@brewhubphl.com\",\"regular\":102,\"overtime\":172,\"sunday_regular\":7,\"sunday_overtime\":0},{\"email\":\"oz23@brewhubphl.com\",\"regular\":778,\"overtime\":555,\"sunday_regular\":79,\"sunday_overtime\":17}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1582323}]},{\"id\":\"many_workers-012\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"pia@brewhubphl.com\":4113,\"nia@brewhubphl.com\":1549,\"ben@brewhubphl.com\":705,\"mo@brewhubphl.com\":315,\"eli@brewhubphl.com\":3204,\"kai@brewhubphl.com\":3514,\"oz8@brewhubphl.com\":2748,\"lu64@brewhubphl.com\":3094,\"eli89@brewhubphl.com\":4009,\"gus@brewhubphl.com\":4016,\"dee39@brewhubphl.com\":3530,\"ana@brewhubphl.com\":308,\"ben28@brewhubphl.com\":2867},\"workers\":[{\"email\":\"pia@brewhubphl.com\",\"regular\":40,\"overtime\":1945,\"sunday_regular\":1982,\"sunday_overtime\":754},{\"email\":\"nia@brewhubphl.com\",\"regular\":213,\"overtime\":623,\"sunday_regular\":310,\"sunday_overtime\":632},{\"email\":\"ben@brewhubphl.com\",\"regular\":798,\"overtime\":6,\"sunday_regular\":5,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":350,\"overtime\":1,\"sunday_regular\":0,\"sunday_overtime\":10},{\"email\":\"eli@brewhubphl.com\",\"regular\":2503,\"overtime\":1072,\"sunday_regular\":25,\"sunday_overtime\":78},{\"email\":\"kai@brewhubphl.com\",\"regular\":2908,\"overtime\":478,\"sunday_regular\":555,\"sunday_overtime\":92},{\"email\":\"oz8@brewhubphl.com\",\"regular\":2148,\"overtime\":771,\"sunday_regular\":196,\"sunday_overtime\":39},{\"email\":\"lu64@brewhubphl.com\",\"regular\":2167,\"overtime\":1306,\"sunday_regular\":46,\"sunday_overtime\":32},{\"email\":\"eli89@brewhubphl.com\",\"regular\":343,\"overtime\":3253,\"sunday_regular\":110,\"sunday_overtime\":896},{\"email\":\"gus@brewhubphl.com\",\"regular\":2315,\"overtime\":60,\"sunday_regular\":1241,\"sunday_overtime\":994},{\"email\":\"dee39@brewhubphl.com\",\"regular\":3618,\"overtime\":434,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":163,\"overtime\":109,\"sunday_regular\":57,\"sunday_overtime\":24},{\"email\":\"ben28@brewhubphl.com\",\"regular\":139,\"overtime\":524,\"sunday_regular\":2578,\"sunday_overtime\":50}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":169862}]},{\"id\":\"many_workers-013\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"gus@brewhubphl.com\":41700,\"mo@brewhubphl.com\":1729,\"lu@brewhubphl.com\":18383,\"ivo@brewhubphl.com\":7600,\"pia@brewhubphl.com\":3995,\"fay@brewhubphl.com\":30337,\"nia84@brewhubphl.com\":27870,\"ben85@brewhubphl.com\":36250,\"jo91@brewhubphl.com\":36493,\"jo16@brewhubphl.com\":35122,\"kai32@brewhubphl.com\":26468,\"mo23@brewhubphl.com\":32192,\"hana93@brewhubphl.com\":44198,\"hana77@brewhubphl.com\":29652},\"workers\":[{\"email\":\"gus@brewhubphl.com\",\"regular\":1787,\"overtime\":1744,\"sunday_regular\":299,\"sunday_overtime\":126},{\"email\":\"mo@brewhubphl.com\",\"regular\":120,\"overtime\":39,\"sunday_regular\":5,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":1122,\"overtime\":105,\"sunday_regular\":181,\"sunday_overtime\":336},{\"email\":\"ivo@brewhubphl.com\",\"regular\":418,\"overtime\":301,\"sunday_regular\":1,\"sunday_overtime\":1},{\"email\":\"pia@brewhubphl.com\",\"regular\":190,\"overtime\":44,\"sunday_regular\":144,\"sunday_overtime\":1},{\"email\":\"fay@brewhubphl.com\",\"regular\":34,\"overtime\":1174,\"sunday_regular\":1380,\"sunday_overtime\":290},{\"email\":\"nia84@brewhubphl.com\",\"regular\":848,\"overtime\":1098,\"sunday_regular\":306,\"sunday_overtime\":392},{\"email\":\"ben85@brewhubphl.com\",\"regular\":382,\"overtime\":1784,\"sunday_regular\":374,\"sunday_overtime\":899},{\"email\":\"jo91@brewhubphl.com\",\"regular\":2320,\"overtime\":1135,\"sunday_regular\":1,\"sunday_overtime\":6},{\"email\":\"jo16@brewhubphl.com\",\"regular\":749,\"overtime\":1314,\"sunday_regular\":1166,\"sunday_overtime\":103},{\"email\":\"kai32@brewhubphl.com\",\"regular\":375,\"overtime\":209,\"sunday_regular\":154,\"sunday_overtime\":1773},{\"email\":\"mo23@brewhubphl.com\",\"regular\":1257,\"overtime\":1263,\"sunday_regular\":106,\"sunday_overtime\":428},{\"email\":\"hana93@brewhubphl.com\",\"regular\":1877,\"overtime\":1106,\"sunday_regular\":874,\"sunday_overtime\":336},{\"email\":\"hana77@brewhubphl.com\",\"regular\":1762,\"overtime\":651,\"sunday_regular\":10,\"sunday_overtime\":390}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1859946}]},{\"id\":\"many_workers-014\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"pia13@brewhubphl.com\":3675,\"hana85@brewhubphl.com\":34122,\"cy@brewhubphl.com\":10803,\"lu@brewhubphl.com\":8255,\"ben41@brewhubphl.com\":7226,\"eli@brewhubphl.com\":14771,\"hana@brewhubphl.com\":38836,\"kai68@brewhubphl.com\":11593,\"ivo29@brewhubphl.com\":37282,\"gus@brewhubphl.com\":32098,\"ana@brewhubphl.com\":21615,\"ben@brewhubphl.com\":15987},\"workers\":[{\"email\":\"pia13@brewhubphl.com\",\"regular\":195,\"overtime\":56,\"sunday_regular\":16,\"sunday_overtime\":147},{\"email\":\"hana85@brewhubphl.com\",\"regular\":581,\"overtime\":2600,\"sunday_regular\":632,\"sunday_overtime\":31},{\"email\":\"cy@brewhubphl.com\",\"regular\":1086,\"overtime\":31,\"sunday_regular\":0,\"sunday_overtime\":100},{\"email\":\"lu@brewhubphl.com\",\"regular\":252,\"overtime\":138,\"sunday_regular\":207,\"sunday_overtime\":333},{\"email\":\"ben41@brewhubphl.com\",\"regular\":57,\"overtime\":153,\"sunday_regular\":388,\"sunday_overtime\":216},{\"email\":\"eli@brewhubphl.com\",\"regular\":25,\"overtime\":1402,\"sunday_regular\":200,\"sunday_overtime\":37},{\"email\":\"hana@brewhubphl.com\",\"regular\":480,\"overtime\":2883,\"sunday_regular\":809,\"sunday_overtime\":203},{\"email\":\"kai68@brewhubphl.com\",\"regular\":150,\"overtime\":1142,\"sunday_regular\":11,\"sunday_overtime\":3},{\"email\":\"ivo29@brewhubphl.com\",\"regular\":4125,\"overtime\":31,\"sunday_regular\":17,\"sunday_overtime\":27},{\"email\":\"gus@brewhubphl.com\",\"regular\":3453,\"overtime\":142,\"sunday_regular\":19,\"sunday_overtime\":2},{\"email\":\"ana@brewhubphl.com\",\"regular\":2160,\"overtime\":37,\"sunday_regular\":90,\"sunday_overtime\":148},{\"email\":\"ben@brewhubphl.com\",\"regular\":214,\"overtime\":205,\"sunday_regular\":718,\"sunday_overtime\":664}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1181315}]},{\"id\":\"many_workers-015\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"oz18@brewhubphl.com\":10439,\"oz@brewhubphl.com\":14496,\"eli98@brewhubphl.com\":5408,\"eli@brewhubphl.com\":18100,\"ivo94@brewhubphl.com\":11319,\"mo@brewhubphl.com\":8648,\"ben84@brewhubphl.com\":10543,\"oz98@brewhubphl.com\":14392,\"pia@brewhubphl.com\":71,\"ben4@brewhubphl.com\":14123,\"jo@brewhubphl.com\":18253,\"ivo@brewhubphl.com\":14454,\"cy@brewhubphl.com\":12165},\"workers\":[{\"email\":\"oz18@brewhubphl.com\",\"regular\":1849,\"overtime\":313,\"sunday_regular\":177,\"sunday_overtime\":178},{\"email\":\"oz@brewhubphl.com\",\"regular\":507,\"overtime\":2757,\"sunday_regular\":48,\"sunday_overtime\":183},{\"email\":\"eli98@brewhubphl.com\",\"regular\":42,\"overtime\":514,\"sunday_regular\":641,\"sunday_overtime\":107},{\"email\":\"eli@brewhubphl.com\",\"regular\":3334,\"overtime\":898,\"sunday_regular\":1,\"sunday_overtime\":131},{\"email\":\"ivo94@brewhubphl.com\",\"regular\":1133,\"overtime\":1471,\"sunday_regular\":49,\"sunday_overtime\":76},{\"email\":\"mo@brewhubphl.com\",\"regular\":1510,\"overtime\":521,\"sunday_regular\":17,\"sunday_overtime\":37},{\"email\":\"ben84@brewhubphl.com\",\"regular\":1214,\"overtime\":780,\"sunday_regular\":397,\"sunday_overtime\":151},{\"email\":\"oz98@brewhubphl.com\",\"regular\":3042,\"overtime\":418,\"sunday_regular\":3,\"sunday_overtime\":7},{\"email\":\"pia@brewhubphl.com\",\"regular\":5,\"overtime\":2,\"sunday_regular\":2,\"sunday_overtime\":8},{\"email\":\"ben4@brewhubphl.com\",\"regular\":3088,\"overtime\":46,\"sunday_regular\":234,\"sunday_overtime\":37},{\"email\":\"jo@brewhubphl.com\",\"regular\":3932,\"overtime\":192,\"sunday_regular\":215,\"sunday_overtime\":62},{\"email\":\"ivo@brewhubphl.com\",\"regular\":1020,\"overtime\":2166,\"sunday_regular\":246,\"sunday_overtime\":53},{\"email\":\"cy@brewhubphl.com\",\"regular\":2543,\"overtime\":116,\"sunday_regular\":34,\"sunday_overtime\":240}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":762058}]},{\"id\":\"many_workers-016\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"cy@brewhubphl.com\":7179,\"cy44@brewhubphl.com\":24593,\"nia@brewhubphl.com\":34757,\"lu48@brewhubphl.com\":10743,\"lu@brewhubphl.com\":34655,\"fay@brewhubphl.com\":33612,\"gus@brewhubphl.com\":20529,\"oz@brewhubphl.com\":21406,\"eli@brewhubphl.com\":20798,\"oz10@brewhubphl.com\":20203,\"pia@brewhubphl.com\":3390,\"mo@brewhubphl.com\":33503,\"lu8@brewhubphl.com\":26209},\"workers\":[{\"email\":\"cy@brewhubphl.com\",\"regular\":755,\"overtime\":147,\"sunday_regular\":17,\"sunday_overtime\":72},{\"email\":\"cy44@brewhubphl.com\",\"regular\":1844,\"overtime\":1067,\"sunday_regular\":420,\"sunday_overtime\":64},{\"email\":\"nia@brewhubphl.com\",\"regular\":2630,\"overtime\":1985,\"sunday_regular\":161,\"sunday_overtime\":22},{\"email\":\"lu48@brewhubphl.com\",\"regular\":966,\"overtime\":109,\"sunday_regular\":113,\"sunday_overtime\":295},{\"email\":\"lu@brewhubphl.com\",\"regular\":2542,\"overtime\":715,\"sunday_regular\":330,\"sunday_overtime\":1197},{\"email\":\"fay@brewhubphl.com\",\"regular\":3269,\"overtime\":496,\"sunday_regular\":250,\"sunday_overtime\":625},{\"email\":\"gus@brewhubphl.com\",\"regular\":1144,\"overtime\":432,\"sunday_regular\":698,\"sunday_overtime\":560},{\"email\":\"oz@brewhubphl.com\",\"regular\":243,\"overtime\":732,\"sunday_regular\":132,\"sunday_overtime\":1848},{\"email\":\"eli@brewhubphl.com\",\"regular\":2086,\"overtime\":654,\"sunday_regular\":51,\"sunday_overtime\":80},{\"email\":\"oz10@brewhubphl.com\",\"regular\":1545,\"overtime\":276,\"sunday_regular\":356,\"sunday_overtime\":612},{\"email\":\"pia@brewhubphl.com\",\"regular\":358,\"overtime\":29,\"sunday_regular\":78,\"sunday_overtime\":3},{\"email\":\"mo@brewhubphl.com\",\"regular\":1423,\"overtime\":2872,\"sunday_regular\":282,\"sunday_overtime\":48},{\"email\":\"lu8@brewhubphl.com\",\"regular\":875,\"overtime\":1592,\"sunday_regular\":380,\"sunday_overtime\":771}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1457886}]},{\"id\":\"many_workers-017\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"cy@brewhubphl.com\":3902,\"lu@brewhubphl.com\":21617,\"oz@brewhubphl.com\":13544,\"kai@brewhubphl.com\":22143,\"dee@brewhubphl.com\":16726,\"dee84@brewhubphl.com\":25175,\"fay93@brewhubphl.com\":419,\"gus@brewhubphl.com\":12458,\"hana@brewhubphl.com\":18553,\"ana@brewhubphl.com\":43706,\"fay@brewhubphl.com\":17124,\"jo61@brewhubphl.com\":4493,\"jo25@brewhubphl.com\":12587,\"mo93@brewhubphl.com\":5310,\"jo@brewhubphl.com\":47275,\"dee25@brewhubphl.com\":35537},\"workers\":[{\"email\":\"cy@brewhubphl.com\",\"regular\":53,\"overtime\":206,\"sunday_regular\":76,\"sunday_overtime\":28},{\"email\":\"lu@brewhubphl.com\",\"regular\":741,\"overtime\":326,\"sunday_regular\":799,\"sunday_overtime\":145},{\"email\":\"oz@brewhubphl.com\",\"regular\":787,\"overtime\":378,\"sunday_regular\":8,\"sunday_overtime\":87},{\"email\":\"kai@brewhubphl.com\",\"regular\":1019,\"overtime\":1026,\"sunday_regular\":3,\"sunday_overtime\":12},{\"email\":\"dee@brewhubphl.com\",\"regular\":1,\"overtime\":1427,\"sunday_regular\":85,\"sunday_overtime\":43},{\"email\":\"dee84@brewhubphl.com\",\"regular\":85,\"overtime\":74,\"sunday_regular\":370,\"sunday_overtime\":1813},{\"email\":\"fay93@brewhubphl.com\",\"regular\":33,\"overtime\":1,\"sunday_regular\":1,\"sunday_overtime\":4},{\"email\":\"gus@brewhubphl.com\",\"regular\":292,\"overtime\":160,\"sunday_regular\":263,\"sunday_overtime\":444},{\"email\":\"hana@brewhubphl.com\",\"regular\":805,\"overtime\":348,\"sunday_regular\":370,\"sunday_overtime\":203},{\"email\":\"ana@brewhubphl.com\",\"regular\":641,\"overtime\":1294,\"sunday_regular\":1514,\"sunday_overtime\":617},{\"email\":\"fay@brewhubphl.com\",\"regular\":1084,\"overtime\":265,\"sunday_regular\":146,\"sunday_overtime\":98},{\"email\":\"jo61@brewhubphl.com\",\"regular\":170,\"overtime\":235,\"sunday_regular\":1,\"sunday_overtime\":12},{\"email\":\"jo25@brewhubphl.com\",\"regular\":1102,\"overtime\":19,\"sunday_regular\":17,\"sunday_overtime\":33},{\"email\":\"mo93@brewhubphl.com\",\"regular\":436,\"overtime\":42,\"sunday_regular\":3,\"sunday_overtime\":13},{\"email\":\"jo@brewhubphl.com\",\"regular\":2119,\"overtime\":845,\"sunday_regular\":1165,\"sunday_overtime\":269},{\"email\":\"dee25@brewhubphl.com\",\"regular\":2992,\"overtime\":86,\"sunday_regular\":96,\"sunday_overtime\":132}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1502846}]},{\"id\":\"many_workers-018\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"jo29@brewhubphl.com\":4710,\"dee@brewhubphl.com\":38653,\"nia@brewhubphl.com\":18476,\"ana@brewhubphl.com\":16104,\"lu@brewhubphl.com\":24590,\"kai40@brewhubphl.com\":18774,\"pia@brewhubphl.com\":17813,\"ivo@brewhubphl.com\":8902,\"gus@brewhubphl.com\":17754,\"dee72@brewhubphl.com\":33943,\"hana@brewhubphl.com\":38568,\"dee53@brewhubphl.com\":3818},\"workers\":[{\"email\":\"jo29@brewhubphl.com\",\"regular\":382,\"overtime\":9,\"sunday_regular\":33,\"sunday_overtime\":130},{\"email\":\"dee@brewhubphl.com\",\"regular\":3406,\"overtime\":134,\"sunday_regular\":897,\"sunday_overtime\":109},{\"email\":\"nia@brewhubphl.com\",\"regular\":115,\"overtime\":1526,\"sunday_regular\":50,\"sunday_overtime\":482},{\"email\":\"ana@brewhubphl.com\",\"regular\":525,\"overtime\":512,\"sunday_regular\":695,\"sunday_overtime\":162},{\"email\":\"lu@brewhubphl.com\",\"regular\":1342,\"overtime\":1188,\"sunday_regular\":266,\"sunday_overtime\":96},{\"email\":\"kai40@brewhubphl.com\",\"regular\":1290,\"overtime\":118,\"sunday_regular\":262,\"sunday_overtime\":538},{\"email\":\"pia@brewhubphl.com\",\"regular\":2055,\"overtime\":12,\"sunday_regular\":12,\"sunday_overtime\":16},{\"email\":\"ivo@brewhubphl.com\",\"regular\":52,\"overtime\":667,\"sunday_regular\":202,\"sunday_overtime\":126},{\"email\":\"gus@brewhubphl.com\",\"regular\":2073,\"overtime\":8,\"sunday_regular\":5,\"sunday_overtime\":2},{\"email\":\"dee72@brewhubphl.com\",\"regular\":2224,\"overtime\":1315,\"sunday_regular\":240,\"sunday_overtime\":213},{\"email\":\"hana@brewhubphl.com\",\"regular\":893,\"overtime\":990,\"sunday_regular\":207,\"sunday_overtime\":2446},{\"email\":\"dee53@brewhubphl.com\",\"regular\":155,\"overtime\":152,\"sunday_regular\":56,\"sunday_overtime\":86}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1210528}]},{\"id\":\"many_workers-019\",\"category\":\"many_workers\",\"tie_decides\":false,\"answer\":{\"kai45@brewhubphl.com\":27655,\"hana@brewhubphl.com\":13302,\"jo@brewhubphl.com\":13772,\"mo@brewhubphl.com\":25013,\"ana@brewhubphl.com\":1111,\"oz2@brewhubphl.com\":34701,\"ben25@brewhubphl.com\":36413,\"gus@brewhubphl.com\":20899,\"ben52@brewhubphl.com\":27915,\"eli@brewhubphl.com\":6476,\"ivo@brewhubphl.com\":2752,\"cy@brewhubphl.com\":30598,\"pia@brewhubphl.com\":43829,\"dee@brewhubphl.com\":37894,\"lu@brewhubphl.com\":24702,\"kai@brewhubphl.com\":38054},\"workers\":[{\"email\":\"kai45@brewhubphl.com\",\"regular\":2293,\"overtime\":22,\"sunday_regular\":115,\"sunday_overtime\":333},{\"email\":\"hana@brewhubphl.com\",\"regular\":721,\"overtime\":146,\"sunday_regular\":113,\"sunday_overtime\":349},{\"email\":\"jo@brewhubphl.com\",\"regular\":1092,\"overtime\":149,\"sunday_regular\":94,\"sunday_overtime\":41},{\"email\":\"mo@brewhubphl.com\",\"regular\":1818,\"overtime\":181,\"sunday_regular\":308,\"sunday_overtime\":192},{\"email\":\"ana@brewhubphl.com\",\"regular\":10,\"overtime\":34,\"sunday_regular\":49,\"sunday_overtime\":18},{\"email\":\"oz2@brewhubphl.com\",\"regular\":1589,\"overtime\":116,\"sunday_regular\":204,\"sunday_overtime\":1558},{\"email\":\"ben25@brewhubphl.com\",\"regular\":1023,\"overtime\":852,\"sunday_regular\":1212,\"sunday_overtime\":551},{\"email\":\"gus@brewhubphl.com\",\"regular\":758,\"overtime\":924,\"sunday_regular\":86,\"sunday_overtime\":320},{\"email\":\"ben52@brewhubphl.com\",\"regular\":205,\"overtime\":222,\"sunday_regular\":1253,\"sunday_overtime\":1109},{\"email\":\"eli@brewhubphl.com\",\"regular\":588,\"overtime\":17,\"sunday_regular\":1,\"sunday_overtime\":41},{\"email\":\"ivo@brewhubphl.com\",\"regular\":75,\"overtime\":2,\"sunday_regular\":24,\"sunday_overtime\":174},{\"email\":\"cy@brewhubphl.com\",\"regular\":2964,\"overtime\":21,\"sunday_regular\":50,\"sunday_overtime\":22},{\"email\":\"pia@brewhubphl.com\",\"regular\":4064,\"overtime\":313,\"sunday_regular\":2,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":1512,\"overtime\":780,\"sunday_regular\":277,\"sunday_overtime\":1217},{\"email\":\"lu@brewhubphl.com\",\"regular\":1260,\"overtime\":626,\"sunday_regular\":474,\"sunday_overtime\":108},{\"email\":\"kai@brewhubphl.com\",\"regular\":1928,\"overtime\":1839,\"sunday_regular\":25,\"sunday_overtime\":10}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":1925430}]},{\"id\":\"funding_filter-000\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"pia@brewhubphl.com\":1699,\"dee@brewhubphl.com\":3964,\"eli@brewhubphl.com\":1922,\"kai38@brewhubphl.com\":2889},\"workers\":[{\"email\":\"pia@brewhubphl.com\",\"regular\":555,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":1295,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli@brewhubphl.com\",\"regular\":628,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai38@brewhubphl.com\",\"regular\":944,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\" Agent_API \",\"subtotal_cents\":14049},{\"source\":\"square_terminal\",\"subtotal_cents\":72051},{\"source\":\"online\",\"subtotal_cents\":19664},{\"source\":\"pos\",\"subtotal_cents\":26805},{\"source\":\"online\",\"subtotal_cents\":-4201},{\"source\":\"ONLINE\",\"subtotal_cents\":18659},{\"source\":null,\"subtotal_cents\":71279}]},{\"id\":\"funding_filter-001\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"eli@brewhubphl.com\":2119,\"mo@brewhubphl.com\":4067,\"hana56@brewhubphl.com\":2923,\"kai@brewhubphl.com\":2852},\"workers\":[{\"email\":\"eli@brewhubphl.com\",\"regular\":1152,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":2211,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana56@brewhubphl.com\",\"regular\":1589,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":1550,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":-3819},{\"source\":\"square_terminal\",\"subtotal_cents\":63746},{\"source\":\"ONLINE\",\"subtotal_cents\":17349},{\"source\":null,\"subtotal_cents\":40643},{\"source\":\"online\",\"subtotal_cents\":33653},{\"source\":\"pos\",\"subtotal_cents\":7719},{\"source\":\" Agent_API \",\"subtotal_cents\":8806}]},{\"id\":\"funding_filter-002\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"kai@brewhubphl.com\":1604,\"hana@brewhubphl.com\":4158},\"workers\":[{\"email\":\"kai@brewhubphl.com\",\"regular\":794,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":2058,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"ONLINE\",\"subtotal_cents\":7057},{\"source\":\"online\",\"subtotal_cents\":-3250},{\"source\":\" Agent_API \",\"subtotal_cents\":15458},{\"source\":\"square_terminal\",\"subtotal_cents\":15195},{\"source\":null,\"subtotal_cents\":51336},{\"source\":\"online\",\"subtotal_cents\":6295},{\"source\":\"pos\",\"subtotal_cents\":81633}]},{\"id\":\"funding_filter-003\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"pia@brewhubphl.com\":6233,\"lu55@brewhubphl.com\":942,\"ben@brewhubphl.com\":4982},\"workers\":[{\"email\":\"pia@brewhubphl.com\",\"regular\":2262,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu55@brewhubphl.com\",\"regular\":342,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":1808,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":38112},{\"source\":\"online\",\"subtotal_cents\":-4251},{\"source\":\"pos\",\"subtotal_cents\":4251},{\"source\":\"ONLINE\",\"subtotal_cents\":11459},{\"source\":\"square_terminal\",\"subtotal_cents\":58187},{\"source\":null,\"subtotal_cents\":22034},{\"source\":\" Agent_API \",\"subtotal_cents\":11218}]},{\"id\":\"funding_filter-004\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"ana@brewhubphl.com\":3671,\"kai94@brewhubphl.com\":2756},\"workers\":[{\"email\":\"ana@brewhubphl.com\",\"regular\":2247,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai94@brewhubphl.com\",\"regular\":1687,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":null,\"subtotal_cents\":3008},{\"source\":\"square_terminal\",\"subtotal_cents\":58587},{\"source\":\"ONLINE\",\"subtotal_cents\":3037},{\"source\":\"online\",\"subtotal_cents\":-742},{\"source\":\" Agent_API \",\"subtotal_cents\":11219},{\"source\":\"pos\",\"subtotal_cents\":19704},{\"source\":\"online\",\"subtotal_cents\":17882}]},{\"id\":\"funding_filter-005\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"eli@brewhubphl.com\":2533,\"pia@brewhubphl.com\":9710},\"workers\":[{\"email\":\"eli@brewhubphl.com\",\"regular\":376,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":1441,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\" Agent_API \",\"subtotal_cents\":8481},{\"source\":\"online\",\"subtotal_cents\":-2892},{\"source\":\"online\",\"subtotal_cents\":47441},{\"source\":\"pos\",\"subtotal_cents\":66359},{\"source\":null,\"subtotal_cents\":44229},{\"source\":\"square_terminal\",\"subtotal_cents\":61300},{\"source\":\"ONLINE\",\"subtotal_cents\":5294}]},{\"id\":\"funding_filter-006\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"fay37@brewhubphl.com\":3941,\"hana@brewhubphl.com\":1412,\"mo41@brewhubphl.com\":1364,\"eli@brewhubphl.com\":3069},\"workers\":[{\"email\":\"fay37@brewhubphl.com\",\"regular\":1988,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":712,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo41@brewhubphl.com\",\"regular\":688,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli@brewhubphl.com\",\"regular\":1548,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"ONLINE\",\"subtotal_cents\":4648},{\"source\":\"online\",\"subtotal_cents\":-4573},{\"source\":null,\"subtotal_cents\":32333},{\"source\":\"pos\",\"subtotal_cents\":76239},{\"source\":\" Agent_API \",\"subtotal_cents\":12109},{\"source\":\"square_terminal\",\"subtotal_cents\":58997},{\"source\":\"online\",\"subtotal_cents\":32176}]},{\"id\":\"funding_filter-007\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"mo59@brewhubphl.com\":2963,\"gus99@brewhubphl.com\":412,\"lu@brewhubphl.com\":1838,\"fay33@brewhubphl.com\":3865},\"workers\":[{\"email\":\"mo59@brewhubphl.com\",\"regular\":1689,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus99@brewhubphl.com\",\"regular\":235,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":1048,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay33@brewhubphl.com\",\"regular\":2203,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":12783},{\"source\":\"square_terminal\",\"subtotal_cents\":10754},{\"source\":\"pos\",\"subtotal_cents\":53398},{\"source\":\"online\",\"subtotal_cents\":-364},{\"source\":null,\"subtotal_cents\":47908},{\"source\":\"ONLINE\",\"subtotal_cents\":16146},{\"source\":\" Agent_API \",\"subtotal_cents\":16463}]},{\"id\":\"funding_filter-008\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"nia@brewhubphl.com\":2524,\"gus99@brewhubphl.com\":2543,\"ben@brewhubphl.com\":830},\"workers\":[{\"email\":\"nia@brewhubphl.com\",\"regular\":2171,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus99@brewhubphl.com\",\"regular\":2187,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":714,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":15559},{\"source\":\"online\",\"subtotal_cents\":-3884},{\"source\":null,\"subtotal_cents\":19640},{\"source\":\"pos\",\"subtotal_cents\":16173},{\"source\":\"ONLINE\",\"subtotal_cents\":8886},{\"source\":\" Agent_API \",\"subtotal_cents\":5040},{\"source\":\"square_terminal\",\"subtotal_cents\":10840}]},{\"id\":\"funding_filter-009\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"fay@brewhubphl.com\":6193,\"lu12@brewhubphl.com\":2346},\"workers\":[{\"email\":\"fay@brewhubphl.com\",\"regular\":1713,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu12@brewhubphl.com\",\"regular\":649,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"pos\",\"subtotal_cents\":38632},{\"source\":\"ONLINE\",\"subtotal_cents\":15091},{\"source\":\"online\",\"subtotal_cents\":15592},{\"source\":\"square_terminal\",\"subtotal_cents\":86930},{\"source\":\"online\",\"subtotal_cents\":-4063},{\"source\":null,\"subtotal_cents\":78982},{\"source\":\" Agent_API \",\"subtotal_cents\":12013}]},{\"id\":\"funding_filter-010\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"gus82@brewhubphl.com\":1848,\"mo43@brewhubphl.com\":1218,\"oz84@brewhubphl.com\":1846,\"kai@brewhubphl.com\":98},\"workers\":[{\"email\":\"gus82@brewhubphl.com\",\"regular\":2259,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo43@brewhubphl.com\",\"regular\":1489,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz84@brewhubphl.com\",\"regular\":2257,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":120,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\" Agent_API \",\"subtotal_cents\":6930},{\"source\":\"pos\",\"subtotal_cents\":83897},{\"source\":\"online\",\"subtotal_cents\":17978},{\"source\":\"square_terminal\",\"subtotal_cents\":52851},{\"source\":\"ONLINE\",\"subtotal_cents\":143},{\"source\":\"online\",\"subtotal_cents\":-2057},{\"source\":null,\"subtotal_cents\":25037}]},{\"id\":\"funding_filter-011\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"nia91@brewhubphl.com\":4350,\"nia22@brewhubphl.com\":5787},\"workers\":[{\"email\":\"nia91@brewhubphl.com\",\"regular\":1725,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia22@brewhubphl.com\",\"regular\":2295,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"ONLINE\",\"subtotal_cents\":14678},{\"source\":\"online\",\"subtotal_cents\":-1453},{\"source\":\"square_terminal\",\"subtotal_cents\":79247},{\"source\":\"online\",\"subtotal_cents\":23414},{\"source\":null,\"subtotal_cents\":84419},{\"source\":\"pos\",\"subtotal_cents\":69449},{\"source\":\" Agent_API \",\"subtotal_cents\":12597}]},{\"id\":\"funding_filter-012\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"dee45@brewhubphl.com\":2714,\"ben52@brewhubphl.com\":259,\"mo@brewhubphl.com\":2761},\"workers\":[{\"email\":\"dee45@brewhubphl.com\",\"regular\":2271,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben52@brewhubphl.com\",\"regular\":217,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":2310,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"ONLINE\",\"subtotal_cents\":14334},{\"source\":\" Agent_API \",\"subtotal_cents\":465},{\"source\":\"online\",\"subtotal_cents\":-2958},{\"source\":\"square_terminal\",\"subtotal_cents\":29632},{\"source\":\"online\",\"subtotal_cents\":13875},{\"source\":\"pos\",\"subtotal_cents\":86244},{\"source\":null,\"subtotal_cents\":30630}]},{\"id\":\"funding_filter-013\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"jo@brewhubphl.com\":2658,\"kai33@brewhubphl.com\":1909},\"workers\":[{\"email\":\"jo@brewhubphl.com\",\"regular\":1908,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai33@brewhubphl.com\",\"regular\":1370,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\" Agent_API \",\"subtotal_cents\":16377},{\"source\":\"square_terminal\",\"subtotal_cents\":48638},{\"source\":\"online\",\"subtotal_cents\":-2830},{\"source\":\"ONLINE\",\"subtotal_cents\":1784},{\"source\":\"online\",\"subtotal_cents\":4674},{\"source\":\"pos\",\"subtotal_cents\":33388},{\"source\":null,\"subtotal_cents\":38218}]},{\"id\":\"funding_filter-014\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"gus90@brewhubphl.com\":4773,\"ana@brewhubphl.com\":1509},\"workers\":[{\"email\":\"gus90@brewhubphl.com\",\"regular\":1961,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana@brewhubphl.com\",\"regular\":620,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":17484},{\"source\":\"ONLINE\",\"subtotal_cents\":11409},{\"source\":\" Agent_API \",\"subtotal_cents\":2521},{\"source\":null,\"subtotal_cents\":78733},{\"source\":\"online\",\"subtotal_cents\":-3630},{\"source\":\"square_terminal\",\"subtotal_cents\":28126},{\"source\":\"pos\",\"subtotal_cents\":6476}]},{\"id\":\"funding_filter-015\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"cy@brewhubphl.com\":5732,\"hana@brewhubphl.com\":3605},\"workers\":[{\"email\":\"cy@brewhubphl.com\",\"regular\":1803,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":1134,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":29690},{\"source\":\"online\",\"subtotal_cents\":-231},{\"source\":\" Agent_API \",\"subtotal_cents\":8685},{\"source\":\"pos\",\"subtotal_cents\":86142},{\"source\":null,\"subtotal_cents\":51592},{\"source\":\"ONLINE\",\"subtotal_cents\":8312},{\"source\":\"square_terminal\",\"subtotal_cents\":2235}]},{\"id\":\"funding_filter-016\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"cy@brewhubphl.com\":287,\"eli63@brewhubphl.com\":4657,\"nia86@brewhubphl.com\":1951},\"workers\":[{\"email\":\"cy@brewhubphl.com\",\"regular\":62,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"eli63@brewhubphl.com\",\"regular\":1007,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia86@brewhubphl.com\",\"regular\":422,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"ONLINE\",\"subtotal_cents\":10917},{\"source\":\" Agent_API \",\"subtotal_cents\":17017},{\"source\":\"online\",\"subtotal_cents\":6542},{\"source\":\"pos\",\"subtotal_cents\":88179},{\"source\":\"square_terminal\",\"subtotal_cents\":57456},{\"source\":null,\"subtotal_cents\":78866},{\"source\":\"online\",\"subtotal_cents\":-3641}]},{\"id\":\"funding_filter-017\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"gus@brewhubphl.com\":2913,\"ana69@brewhubphl.com\":2753,\"pia49@brewhubphl.com\":3176,\"ana80@brewhubphl.com\":935,\"nia@brewhubphl.com\":186},\"workers\":[{\"email\":\"gus@brewhubphl.com\",\"regular\":2051,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana69@brewhubphl.com\",\"regular\":1938,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia49@brewhubphl.com\",\"regular\":2236,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ana80@brewhubphl.com\",\"regular\":658,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia@brewhubphl.com\",\"regular\":131,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"square_terminal\",\"subtotal_cents\":27982},{\"source\":\"online\",\"subtotal_cents\":-1882},{\"source\":\"ONLINE\",\"subtotal_cents\":17452},{\"source\":null,\"subtotal_cents\":9989},{\"source\":\" Agent_API \",\"subtotal_cents\":3459},{\"source\":\"online\",\"subtotal_cents\":28907},{\"source\":\"pos\",\"subtotal_cents\":45452}]},{\"id\":\"funding_filter-018\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"nia45@brewhubphl.com\":2106,\"fay@brewhubphl.com\":4330,\"nia36@brewhubphl.com\":5830},\"workers\":[{\"email\":\"nia45@brewhubphl.com\",\"regular\":751,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":1544,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia36@brewhubphl.com\",\"regular\":2079,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":-2851},{\"source\":\"square_terminal\",\"subtotal_cents\":58079},{\"source\":\"online\",\"subtotal_cents\":44182},{\"source\":\"ONLINE\",\"subtotal_cents\":5400},{\"source\":\"pos\",\"subtotal_cents\":75329},{\"source\":\" Agent_API \",\"subtotal_cents\":11748},{\"source\":null,\"subtotal_cents\":52478}]},{\"id\":\"funding_filter-019\",\"category\":\"funding_filter\",\"tie_decides\":false,\"answer\":{\"jo38@brewhubphl.com\":3295,\"kai@brewhubphl.com\":9983},\"workers\":[{\"email\":\"jo38@brewhubphl.com\",\"regular\":306,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai@brewhubphl.com\",\"regular\":927,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":null,\"subtotal_cents\":22262},{\"source\":\"square_terminal\",\"subtotal_cents\":80103},{\"source\":\" Agent_API \",\"subtotal_cents\":12424},{\"source\":\"online\",\"subtotal_cents\":45963},{\"source\":\"pos\",\"subtotal_cents\":61534},{\"source\":\"online\",\"subtotal_cents\":-2906},{\"source\":\"ONLINE\",\"subtotal_cents\":8007}]},{\"id\":\"bucket_split-000\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"fay3@brewhubphl.com\":7686,\"mo57@brewhubphl.com\":4225,\"gus44@brewhubphl.com\":8187,\"cy@brewhubphl.com\":2089,\"hana@brewhubphl.com\":7714},\"workers\":[{\"email\":\"fay3@brewhubphl.com\",\"regular\":1182,\"overtime\":756,\"sunday_regular\":117,\"sunday_overtime\":186},{\"email\":\"mo57@brewhubphl.com\",\"regular\":867,\"overtime\":23,\"sunday_regular\":249,\"sunday_overtime\":93},{\"email\":\"gus44@brewhubphl.com\",\"regular\":1682,\"overtime\":444,\"sunday_regular\":163,\"sunday_overtime\":98},{\"email\":\"cy@brewhubphl.com\",\"regular\":588,\"overtime\":12,\"sunday_regular\":9,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":270,\"overtime\":1016,\"sunday_regular\":139,\"sunday_overtime\":824}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":149505}]},{\"id\":\"bucket_split-001\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"gus@brewhubphl.com\":7768,\"cy@brewhubphl.com\":9171,\"oz27@brewhubphl.com\":6873,\"jo40@brewhubphl.com\":14911},\"workers\":[{\"email\":\"gus@brewhubphl.com\",\"regular\":358,\"overtime\":709,\"sunday_regular\":163,\"sunday_overtime\":11},{\"email\":\"cy@brewhubphl.com\",\"regular\":765,\"overtime\":316,\"sunday_regular\":293,\"sunday_overtime\":91},{\"email\":\"oz27@brewhubphl.com\",\"regular\":136,\"overtime\":587,\"sunday_regular\":78,\"sunday_overtime\":297},{\"email\":\"jo40@brewhubphl.com\",\"regular\":668,\"overtime\":869,\"sunday_regular\":507,\"sunday_overtime\":338}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":193619}]},{\"id\":\"bucket_split-002\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"eli@brewhubphl.com\":4218,\"kai36@brewhubphl.com\":1059,\"cy89@brewhubphl.com\":940,\"hana@brewhubphl.com\":4905,\"mo51@brewhubphl.com\":6096},\"workers\":[{\"email\":\"eli@brewhubphl.com\",\"regular\":554,\"overtime\":295,\"sunday_regular\":251,\"sunday_overtime\":851},{\"email\":\"kai36@brewhubphl.com\",\"regular\":165,\"overtime\":43,\"sunday_regular\":208,\"sunday_overtime\":74},{\"email\":\"cy89@brewhubphl.com\",\"regular\":167,\"overtime\":197,\"sunday_regular\":1,\"sunday_overtime\":70},{\"email\":\"hana@brewhubphl.com\",\"regular\":1503,\"overtime\":659,\"sunday_regular\":37,\"sunday_overtime\":70},{\"email\":\"mo51@brewhubphl.com\",\"regular\":2531,\"overtime\":187,\"sunday_regular\":88,\"sunday_overtime\":14}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":86094}]},{\"id\":\"bucket_split-003\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"kai47@brewhubphl.com\":4547,\"dee@brewhubphl.com\":1101,\"lu@brewhubphl.com\":2836,\"gus@brewhubphl.com\":456,\"nia@brewhubphl.com\":8003},\"workers\":[{\"email\":\"kai47@brewhubphl.com\",\"regular\":1051,\"overtime\":22,\"sunday_regular\":49,\"sunday_overtime\":154},{\"email\":\"dee@brewhubphl.com\",\"regular\":54,\"overtime\":97,\"sunday_regular\":4,\"sunday_overtime\":154},{\"email\":\"lu@brewhubphl.com\",\"regular\":545,\"overtime\":65,\"sunday_regular\":21,\"sunday_overtime\":165},{\"email\":\"gus@brewhubphl.com\",\"regular\":19,\"overtime\":10,\"sunday_regular\":1,\"sunday_overtime\":98},{\"email\":\"nia@brewhubphl.com\",\"regular\":1639,\"overtime\":126,\"sunday_regular\":481,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":84719}]},{\"id\":\"bucket_split-004\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"lu@brewhubphl.com\":5253,\"cy@brewhubphl.com\":10548,\"hana15@brewhubphl.com\":2032,\"ana@brewhubphl.com\":10482},\"workers\":[{\"email\":\"lu@brewhubphl.com\",\"regular\":606,\"overtime\":90,\"sunday_regular\":147,\"sunday_overtime\":31},{\"email\":\"cy@brewhubphl.com\",\"regular\":1372,\"overtime\":223,\"sunday_regular\":19,\"sunday_overtime\":141},{\"email\":\"hana15@brewhubphl.com\",\"regular\":322,\"overtime\":10,\"sunday_regular\":1,\"sunday_overtime\":5},{\"email\":\"ana@brewhubphl.com\",\"regular\":1415,\"overtime\":88,\"sunday_regular\":145,\"sunday_overtime\":96}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":141579}]},{\"id\":\"bucket_split-005\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"kai29@brewhubphl.com\":6693,\"mo24@brewhubphl.com\":10465,\"cy@brewhubphl.com\":12574,\"nia@brewhubphl.com\":3884,\"dee33@brewhubphl.com\":1102,\"gus23@brewhubphl.com\":10292},\"workers\":[{\"email\":\"kai29@brewhubphl.com\",\"regular\":599,\"overtime\":163,\"sunday_regular\":320,\"sunday_overtime\":467},{\"email\":\"mo24@brewhubphl.com\",\"regular\":210,\"overtime\":425,\"sunday_regular\":1431,\"sunday_overtime\":356},{\"email\":\"cy@brewhubphl.com\",\"regular\":2814,\"overtime\":13,\"sunday_regular\":51,\"sunday_overtime\":32},{\"email\":\"nia@brewhubphl.com\",\"regular\":211,\"overtime\":357,\"sunday_regular\":142,\"sunday_overtime\":189},{\"email\":\"dee33@brewhubphl.com\",\"regular\":151,\"overtime\":88,\"sunday_regular\":10,\"sunday_overtime\":6},{\"email\":\"gus23@brewhubphl.com\",\"regular\":920,\"overtime\":1444,\"sunday_regular\":11,\"sunday_overtime\":7}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":225050}]},{\"id\":\"bucket_split-006\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"nia45@brewhubphl.com\":5345,\"kai33@brewhubphl.com\":9853,\"ivo78@brewhubphl.com\":4039,\"lu@brewhubphl.com\":8375,\"oz@brewhubphl.com\":10098,\"hana46@brewhubphl.com\":1443},\"workers\":[{\"email\":\"nia45@brewhubphl.com\",\"regular\":62,\"overtime\":583,\"sunday_regular\":488,\"sunday_overtime\":201},{\"email\":\"kai33@brewhubphl.com\",\"regular\":270,\"overtime\":1499,\"sunday_regular\":175,\"sunday_overtime\":515},{\"email\":\"ivo78@brewhubphl.com\",\"regular\":422,\"overtime\":173,\"sunday_regular\":108,\"sunday_overtime\":305},{\"email\":\"lu@brewhubphl.com\",\"regular\":2088,\"overtime\":1,\"sunday_regular\":0,\"sunday_overtime\":1},{\"email\":\"oz@brewhubphl.com\",\"regular\":2023,\"overtime\":8,\"sunday_regular\":291,\"sunday_overtime\":198},{\"email\":\"hana46@brewhubphl.com\",\"regular\":27,\"overtime\":148,\"sunday_regular\":107,\"sunday_overtime\":78}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":195766}]},{\"id\":\"bucket_split-007\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"lu56@brewhubphl.com\":1769,\"cy@brewhubphl.com\":2322,\"lu78@brewhubphl.com\":5422},\"workers\":[{\"email\":\"lu56@brewhubphl.com\",\"regular\":732,\"overtime\":94,\"sunday_regular\":3,\"sunday_overtime\":57},{\"email\":\"cy@brewhubphl.com\",\"regular\":778,\"overtime\":166,\"sunday_regular\":104,\"sunday_overtime\":115},{\"email\":\"lu78@brewhubphl.com\",\"regular\":1462,\"overtime\":108,\"sunday_regular\":1119,\"sunday_overtime\":27}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":47567}]},{\"id\":\"bucket_split-008\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"gus37@brewhubphl.com\":1894,\"hana@brewhubphl.com\":8484,\"nia97@brewhubphl.com\":12266},\"workers\":[{\"email\":\"gus37@brewhubphl.com\",\"regular\":170,\"overtime\":64,\"sunday_regular\":146,\"sunday_overtime\":16},{\"email\":\"hana@brewhubphl.com\",\"regular\":545,\"overtime\":485,\"sunday_regular\":542,\"sunday_overtime\":202},{\"email\":\"nia97@brewhubphl.com\",\"regular\":390,\"overtime\":1054,\"sunday_regular\":957,\"sunday_overtime\":164}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":113221}]},{\"id\":\"bucket_split-009\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"hana39@brewhubphl.com\":9014,\"gus65@brewhubphl.com\":7224,\"cy96@brewhubphl.com\":27295,\"oz@brewhubphl.com\":15575,\"gus@brewhubphl.com\":6223,\"jo@brewhubphl.com\":3412},\"workers\":[{\"email\":\"hana39@brewhubphl.com\",\"regular\":32,\"overtime\":690,\"sunday_regular\":44,\"sunday_overtime\":90},{\"email\":\"gus65@brewhubphl.com\",\"regular\":402,\"overtime\":147,\"sunday_regular\":88,\"sunday_overtime\":49},{\"email\":\"cy96@brewhubphl.com\",\"regular\":840,\"overtime\":550,\"sunday_regular\":1034,\"sunday_overtime\":168},{\"email\":\"oz@brewhubphl.com\",\"regular\":32,\"overtime\":471,\"sunday_regular\":706,\"sunday_overtime\":270},{\"email\":\"gus@brewhubphl.com\",\"regular\":230,\"overtime\":353,\"sunday_regular\":2,\"sunday_overtime\":6},{\"email\":\"jo@brewhubphl.com\",\"regular\":158,\"overtime\":48,\"sunday_regular\":11,\"sunday_overtime\":107}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":343716}]},{\"id\":\"bucket_split-010\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"cy33@brewhubphl.com\":1801,\"oz35@brewhubphl.com\":5875,\"kai@brewhubphl.com\":10122,\"jo98@brewhubphl.com\":254,\"pia@brewhubphl.com\":6747},\"workers\":[{\"email\":\"cy33@brewhubphl.com\",\"regular\":446,\"overtime\":29,\"sunday_regular\":10,\"sunday_overtime\":5},{\"email\":\"oz35@brewhubphl.com\",\"regular\":272,\"overtime\":880,\"sunday_regular\":124,\"sunday_overtime\":322},{\"email\":\"kai@brewhubphl.com\",\"regular\":1166,\"overtime\":292,\"sunday_regular\":759,\"sunday_overtime\":536},{\"email\":\"jo98@brewhubphl.com\",\"regular\":4,\"overtime\":9,\"sunday_regular\":1,\"sunday_overtime\":55},{\"email\":\"pia@brewhubphl.com\",\"regular\":1573,\"overtime\":161,\"sunday_regular\":80,\"sunday_overtime\":21}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":123996}]},{\"id\":\"bucket_split-011\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"hana35@brewhubphl.com\":1597,\"cy@brewhubphl.com\":4647,\"jo@brewhubphl.com\":686,\"pia@brewhubphl.com\":4423,\"ben@brewhubphl.com\":7006,\"lu@brewhubphl.com\":466},\"workers\":[{\"email\":\"hana35@brewhubphl.com\",\"regular\":306,\"overtime\":83,\"sunday_regular\":12,\"sunday_overtime\":246},{\"email\":\"cy@brewhubphl.com\",\"regular\":910,\"overtime\":289,\"sunday_regular\":336,\"sunday_overtime\":348},{\"email\":\"jo@brewhubphl.com\",\"regular\":41,\"overtime\":217,\"sunday_regular\":3,\"sunday_overtime\":17},{\"email\":\"pia@brewhubphl.com\",\"regular\":537,\"overtime\":1104,\"sunday_regular\":123,\"sunday_overtime\":28},{\"email\":\"ben@brewhubphl.com\",\"regular\":2290,\"overtime\":151,\"sunday_regular\":153,\"sunday_overtime\":245},{\"email\":\"lu@brewhubphl.com\",\"regular\":53,\"overtime\":60,\"sunday_regular\":13,\"sunday_overtime\":63}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":94129}]},{\"id\":\"bucket_split-012\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"nia@brewhubphl.com\":2102,\"cy69@brewhubphl.com\":1935,\"ben23@brewhubphl.com\":662,\"fay@brewhubphl.com\":234},\"workers\":[{\"email\":\"nia@brewhubphl.com\",\"regular\":1571,\"overtime\":7,\"sunday_regular\":101,\"sunday_overtime\":123},{\"email\":\"cy69@brewhubphl.com\",\"regular\":1016,\"overtime\":129,\"sunday_regular\":139,\"sunday_overtime\":375},{\"email\":\"ben23@brewhubphl.com\",\"regular\":112,\"overtime\":250,\"sunday_regular\":167,\"sunday_overtime\":39},{\"email\":\"fay@brewhubphl.com\",\"regular\":145,\"overtime\":22,\"sunday_regular\":6,\"sunday_overtime\":28}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":24668}]},{\"id\":\"bucket_split-013\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"ben85@brewhubphl.com\":10900,\"oz1@brewhubphl.com\":1298,\"nia35@brewhubphl.com\":15329,\"hana@brewhubphl.com\":15736,\"ben20@brewhubphl.com\":6334},\"workers\":[{\"email\":\"ben85@brewhubphl.com\",\"regular\":542,\"overtime\":314,\"sunday_regular\":623,\"sunday_overtime\":209},{\"email\":\"oz1@brewhubphl.com\",\"regular\":98,\"overtime\":60,\"sunday_regular\":34,\"sunday_overtime\":9},{\"email\":\"nia35@brewhubphl.com\",\"regular\":5,\"overtime\":306,\"sunday_regular\":89,\"sunday_overtime\":1974},{\"email\":\"hana@brewhubphl.com\",\"regular\":813,\"overtime\":335,\"sunday_regular\":435,\"sunday_overtime\":854},{\"email\":\"ben20@brewhubphl.com\",\"regular\":710,\"overtime\":1,\"sunday_regular\":192,\"sunday_overtime\":78}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":247988}]},{\"id\":\"bucket_split-014\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"cy93@brewhubphl.com\":45234,\"ana@brewhubphl.com\":10618,\"eli@brewhubphl.com\":19732},\"workers\":[{\"email\":\"cy93@brewhubphl.com\",\"regular\":670,\"overtime\":1477,\"sunday_regular\":625,\"sunday_overtime\":27},{\"email\":\"ana@brewhubphl.com\",\"regular\":337,\"overtime\":248,\"sunday_regular\":57,\"sunday_overtime\":15},{\"email\":\"eli@brewhubphl.com\",\"regular\":782,\"overtime\":400,\"sunday_regular\":3,\"sunday_overtime\":36}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":377924}]},{\"id\":\"bucket_split-015\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"ben51@brewhubphl.com\":2855,\"ben@brewhubphl.com\":1599,\"jo@brewhubphl.com\":27170,\"gus@brewhubphl.com\":8109,\"eli54@brewhubphl.com\":11719},\"workers\":[{\"email\":\"ben51@brewhubphl.com\",\"regular\":223,\"overtime\":6,\"sunday_regular\":14,\"sunday_overtime\":14},{\"email\":\"ben@brewhubphl.com\",\"regular\":62,\"overtime\":40,\"sunday_regular\":23,\"sunday_overtime\":19},{\"email\":\"jo@brewhubphl.com\",\"regular\":169,\"overtime\":1338,\"sunday_regular\":448,\"sunday_overtime\":491},{\"email\":\"gus@brewhubphl.com\",\"regular\":419,\"overtime\":277,\"sunday_regular\":24,\"sunday_overtime\":10},{\"email\":\"eli54@brewhubphl.com\",\"regular\":1013,\"overtime\":4,\"sunday_regular\":31,\"sunday_overtime\":7}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":257260}]},{\"id\":\"bucket_split-016\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"fay52@brewhubphl.com\":19108,\"mo37@brewhubphl.com\":15647,\"mo@brewhubphl.com\":4599,\"lu@brewhubphl.com\":15390},\"workers\":[{\"email\":\"fay52@brewhubphl.com\",\"regular\":1455,\"overtime\":49,\"sunday_regular\":1360,\"sunday_overtime\":40},{\"email\":\"mo37@brewhubphl.com\",\"regular\":1567,\"overtime\":768,\"sunday_regular\":38,\"sunday_overtime\":5},{\"email\":\"mo@brewhubphl.com\",\"regular\":369,\"overtime\":204,\"sunday_regular\":54,\"sunday_overtime\":72},{\"email\":\"lu@brewhubphl.com\",\"regular\":1523,\"overtime\":540,\"sunday_regular\":269,\"sunday_overtime\":7}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":273720}]},{\"id\":\"bucket_split-017\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"fay2@brewhubphl.com\":23725,\"fay@brewhubphl.com\":10704,\"kai@brewhubphl.com\":9302},\"workers\":[{\"email\":\"fay2@brewhubphl.com\",\"regular\":1832,\"overtime\":8,\"sunday_regular\":45,\"sunday_overtime\":10},{\"email\":\"fay@brewhubphl.com\",\"regular\":57,\"overtime\":385,\"sunday_regular\":380,\"sunday_overtime\":33},{\"email\":\"kai@brewhubphl.com\",\"regular\":324,\"overtime\":101,\"sunday_regular\":197,\"sunday_overtime\":121}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":218655}]},{\"id\":\"bucket_split-018\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"ben@brewhubphl.com\":4487,\"cy@brewhubphl.com\":6090,\"nia@brewhubphl.com\":5460},\"workers\":[{\"email\":\"ben@brewhubphl.com\",\"regular\":805,\"overtime\":1135,\"sunday_regular\":53,\"sunday_overtime\":16},{\"email\":\"cy@brewhubphl.com\",\"regular\":1578,\"overtime\":468,\"sunday_regular\":477,\"sunday_overtime\":204},{\"email\":\"nia@brewhubphl.com\",\"regular\":1448,\"overtime\":900,\"sunday_regular\":17,\"sunday_overtime\":80}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":80188}]},{\"id\":\"bucket_split-019\",\"category\":\"bucket_split\",\"tie_decides\":false,\"answer\":{\"ben79@brewhubphl.com\":6241,\"jo83@brewhubphl.com\":2664,\"gus7@brewhubphl.com\":2899,\"mo@brewhubphl.com\":3845},\"workers\":[{\"email\":\"ben79@brewhubphl.com\",\"regular\":1022,\"overtime\":1136,\"sunday_regular\":12,\"sunday_overtime\":13},{\"email\":\"jo83@brewhubphl.com\",\"regular\":356,\"overtime\":96,\"sunday_regular\":413,\"sunday_overtime\":67},{\"email\":\"gus7@brewhubphl.com\",\"regular\":403,\"overtime\":195,\"sunday_regular\":385,\"sunday_overtime\":31},{\"email\":\"mo@brewhubphl.com\",\"regular\":997,\"overtime\":191,\"sunday_regular\":25,\"sunday_overtime\":132}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":78249}]},{\"id\":\"empty_pool-000\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"nia@brewhubphl.com\":0,\"jo@brewhubphl.com\":0,\"dee@brewhubphl.com\":0},\"workers\":[{\"email\":\"nia@brewhubphl.com\",\"regular\":380,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":267,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":546,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[]},{\"id\":\"empty_pool-001\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"gus@brewhubphl.com\":0,\"fay78@brewhubphl.com\":0,\"pia61@brewhubphl.com\":0,\"cy@brewhubphl.com\":0},\"workers\":[{\"email\":\"gus@brewhubphl.com\",\"regular\":82,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay78@brewhubphl.com\",\"regular\":125,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia61@brewhubphl.com\",\"regular\":371,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy@brewhubphl.com\",\"regular\":335,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":4}]},{\"id\":\"empty_pool-002\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"hana3@brewhubphl.com\":0,\"nia@brewhubphl.com\":0},\"workers\":[{\"email\":\"hana3@brewhubphl.com\",\"regular\":178,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia@brewhubphl.com\",\"regular\":582,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":4}]},{\"id\":\"empty_pool-003\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"mo@brewhubphl.com\":0,\"hana67@brewhubphl.com\":0,\"fay@brewhubphl.com\":0,\"pia@brewhubphl.com\":0},\"workers\":[{\"email\":\"mo@brewhubphl.com\",\"regular\":548,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana67@brewhubphl.com\",\"regular\":105,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":430,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":269,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"pos\",\"subtotal_cents\":50000}]},{\"id\":\"empty_pool-004\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"ana@brewhubphl.com\":0,\"kai76@brewhubphl.com\":0,\"cy73@brewhubphl.com\":0},\"workers\":[{\"email\":\"ana@brewhubphl.com\",\"regular\":248,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai76@brewhubphl.com\",\"regular\":366,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy73@brewhubphl.com\",\"regular\":354,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"pos\",\"subtotal_cents\":50000}]},{\"id\":\"empty_pool-005\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"jo41@brewhubphl.com\":0,\"ivo60@brewhubphl.com\":0,\"nia53@brewhubphl.com\":0},\"workers\":[{\"email\":\"jo41@brewhubphl.com\",\"regular\":272,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo60@brewhubphl.com\",\"regular\":227,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia53@brewhubphl.com\",\"regular\":167,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[]},{\"id\":\"empty_pool-006\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"mo@brewhubphl.com\":0,\"ben47@brewhubphl.com\":0,\"cy92@brewhubphl.com\":0,\"ben@brewhubphl.com\":0},\"workers\":[{\"email\":\"mo@brewhubphl.com\",\"regular\":477,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben47@brewhubphl.com\",\"regular\":156,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy92@brewhubphl.com\",\"regular\":314,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":89,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"pos\",\"subtotal_cents\":50000}]},{\"id\":\"empty_pool-007\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"ivo@brewhubphl.com\":0,\"dee@brewhubphl.com\":0},\"workers\":[{\"email\":\"ivo@brewhubphl.com\",\"regular\":345,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"dee@brewhubphl.com\",\"regular\":323,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"pos\",\"subtotal_cents\":50000}]},{\"id\":\"empty_pool-008\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"ivo90@brewhubphl.com\":0,\"mo@brewhubphl.com\":0,\"kai52@brewhubphl.com\":0,\"oz@brewhubphl.com\":0},\"workers\":[{\"email\":\"ivo90@brewhubphl.com\",\"regular\":91,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":65,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai52@brewhubphl.com\",\"regular\":528,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":193,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":4}]},{\"id\":\"empty_pool-009\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"lu81@brewhubphl.com\":0,\"jo@brewhubphl.com\":0},\"workers\":[{\"email\":\"lu81@brewhubphl.com\",\"regular\":327,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":566,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[]},{\"id\":\"empty_pool-010\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"eli@brewhubphl.com\":0,\"ben@brewhubphl.com\":0,\"oz@brewhubphl.com\":0,\"jo@brewhubphl.com\":0},\"workers\":[{\"email\":\"eli@brewhubphl.com\",\"regular\":252,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":189,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":543,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":539,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":4}]},{\"id\":\"empty_pool-011\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"dee@brewhubphl.com\":0,\"ben@brewhubphl.com\":0,\"pia29@brewhubphl.com\":0,\"pia@brewhubphl.com\":0},\"workers\":[{\"email\":\"dee@brewhubphl.com\",\"regular\":151,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":220,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia29@brewhubphl.com\",\"regular\":504,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"pia@brewhubphl.com\",\"regular\":590,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"pos\",\"subtotal_cents\":50000}]},{\"id\":\"empty_pool-012\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"ben55@brewhubphl.com\":0,\"nia15@brewhubphl.com\":0,\"gus@brewhubphl.com\":0,\"fay90@brewhubphl.com\":0},\"workers\":[{\"email\":\"ben55@brewhubphl.com\",\"regular\":505,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"nia15@brewhubphl.com\",\"regular\":506,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus@brewhubphl.com\",\"regular\":556,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay90@brewhubphl.com\",\"regular\":338,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[]},{\"id\":\"empty_pool-013\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"nia@brewhubphl.com\":0,\"ben@brewhubphl.com\":0,\"fay@brewhubphl.com\":0,\"mo@brewhubphl.com\":0},\"workers\":[{\"email\":\"nia@brewhubphl.com\",\"regular\":71,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben@brewhubphl.com\",\"regular\":190,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":266,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"mo@brewhubphl.com\",\"regular\":447,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[]},{\"id\":\"empty_pool-014\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"cy18@brewhubphl.com\":0,\"gus95@brewhubphl.com\":0,\"oz@brewhubphl.com\":0},\"workers\":[{\"email\":\"cy18@brewhubphl.com\",\"regular\":562,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"gus95@brewhubphl.com\",\"regular\":508,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"oz@brewhubphl.com\",\"regular\":501,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"pos\",\"subtotal_cents\":50000}]},{\"id\":\"empty_pool-015\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"eli79@brewhubphl.com\":0,\"cy90@brewhubphl.com\":0,\"jo@brewhubphl.com\":0,\"ben60@brewhubphl.com\":0},\"workers\":[{\"email\":\"eli79@brewhubphl.com\",\"regular\":206,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"cy90@brewhubphl.com\",\"regular\":74,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":325,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ben60@brewhubphl.com\",\"regular\":392,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[]},{\"id\":\"empty_pool-016\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"cy@brewhubphl.com\":0,\"jo@brewhubphl.com\":0,\"ivo48@brewhubphl.com\":0},\"workers\":[{\"email\":\"cy@brewhubphl.com\",\"regular\":173,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"jo@brewhubphl.com\",\"regular\":97,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"ivo48@brewhubphl.com\",\"regular\":199,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"pos\",\"subtotal_cents\":50000}]},{\"id\":\"empty_pool-017\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"kai@brewhubphl.com\":0,\"kai72@brewhubphl.com\":0},\"workers\":[{\"email\":\"kai@brewhubphl.com\",\"regular\":537,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"kai72@brewhubphl.com\",\"regular\":177,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"online\",\"subtotal_cents\":4}]},{\"id\":\"empty_pool-018\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"nia@brewhubphl.com\":0,\"lu@brewhubphl.com\":0,\"hana@brewhubphl.com\":0,\"fay90@brewhubphl.com\":0},\"workers\":[{\"email\":\"nia@brewhubphl.com\",\"regular\":394,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"lu@brewhubphl.com\",\"regular\":492,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"hana@brewhubphl.com\",\"regular\":183,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay90@brewhubphl.com\",\"regular\":313,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"pos\",\"subtotal_cents\":50000}]},{\"id\":\"empty_pool-019\",\"category\":\"empty_pool\",\"tie_decides\":false,\"answer\":{\"cy@brewhubphl.com\":0,\"fay@brewhubphl.com\":0},\"workers\":[{\"email\":\"cy@brewhubphl.com\",\"regular\":250,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0},{\"email\":\"fay@brewhubphl.com\",\"regular\":453,\"overtime\":0,\"sunday_regular\":0,\"sunday_overtime\":0}],\"orders\":[{\"source\":\"pos\",\"subtotal_cents\":50000}]}]")
for c in CASES:
    c["prompt"] = build_prompt(RULES, c["workers"], c["orders"])
    assert allocate(c["workers"], c["orders"]) == c["answer"], c["id"]  # answer key self-check
BY_ID = {c["id"]: c for c in CASES}
# Cap output tokens. The model proxy reserves quota per call from the max, so the default
# 64k reservation (~$1 on a frontier model) can refuse every call when daily quota runs low.
# 32k still leaves room for the longest answer seen (~17k with reasoning); a truncated
# answer is unparseable and fails the case, which is the right outcome.
MAX_TOKENS = {"max_tokens": 32768}
print(len(CASES), "cases,", sum(c["tie_decides"] for c in CASES), "tie-decided")

In [ ]:
@kbench.task(name="franklin_pool_case", store_task=False)
def solve_case(llm, case_id: str) -> dict:
    c = BY_ID[case_id]
    response = llm.prompt(c["prompt"], extra_api_params=MAX_TOKENS)
    got = parse_output(str(response))
    passed, sums = grade(c["answer"], got)
    kbench.assertions.assert_true(passed, expectation="Every worker's cents match the answer key exactly.")
    return {
        "id": c["id"], "category": c["category"], "tie_decides": c["tie_decides"],
        "pass": passed, "parsed": got is not None, "sums_to_pool": sums,
    }

In [ ]:
# Local smoke test: FRANKLIN_CASE_LIMIT=N runs an evenly spaced N-case subset
# (cases are grouped 20 per category, so N=11 gives one per category). Unset on Kaggle.
LIMIT = int(os.environ.get("FRANKLIN_CASE_LIMIT") or 0)
EVAL_CASES = CASES[:: -(-len(CASES) // LIMIT)][:LIMIT] if LIMIT else CASES
# Index by case id: the SDK keys its run cache by the row label, so retries and local
# re-runs hit the right case instead of whatever sat at that position last time.
EVAL = pd.DataFrame({"case_id": [c["id"] for c in EVAL_CASES]}, index=[c["id"] for c in EVAL_CASES])


@kbench.task(name="Franklin Pool: tip-split payout math")
def franklin_pool(llm) -> float:
    """Share of the 220 cases where every worker's cents are exactly right. Errored runs count as failures."""
    # Nested evaluate() allows one attempt, so retry cases that errored (API
    # timeouts, rate limits) here; the cache skips cases already answered. Wait
    # between rounds so rate limits and quota reservations have time to recover.
    results, pending = {}, EVAL
    with kbench.client.enable_cache():
        for attempt in range(3):
            if attempt:
                print(f"retrying {len(pending)} errored case(s) after a {60 * attempt}s pause")
                time.sleep(60 * attempt)
            runs = solve_case.evaluate(
                llm=[llm], evaluation_data=pending, n_jobs=4, timeout=600, on_failure="continue",
            )
            results.update((r.result["id"], r.result) for r in runs.completed_runs)
            pending = EVAL[~EVAL.case_id.isin(results)]
            if pending.empty:
                break
    rows = pd.DataFrame(list(results.values()))
    errored = len(EVAL_CASES) - len(rows)
    if rows.empty:
        print(f"every case errored ({errored}); see a per-case run for the API error")
        return 0.0
    by_cat = rows.groupby("category")["pass"].agg(["sum", "count"])
    print(by_cat.to_string())
    tie = rows[rows.tie_decides]
    print(f"tie-decided: {int(tie['pass'].sum())}/{len(tie)}")
    print(f"unparseable: {int((~rows.parsed).sum())}  sum != pool: {int((rows.parsed & ~rows.sums_to_pool).sum())}  errored: {errored}")
    return float(rows["pass"].sum()) / len(EVAL_CASES)


run = franklin_pool.run(kbench.llm)
run

In [ ]:
%choose franklin_pool